# LAB - Prompt Engineering Lab
delivered by Carolina, november 2026

**Scenario** 
TechFlow Solutions has hired you as an AI consultant. Their customer service chatbot gives inconsistent answers. You need to work out why and fix the prompts for three tasks:

1. **Sentiment Analysis**: Classify customer messages as positive, negative, or neutral
2. **Product Description Generation**: Create compelling product descriptions from basic product information
3. **Data Extraction**: Extract structured information from unstructured customer feedback

**Success target**: your final prompts give the same result in more than 80% of 15 runs. 

In [1]:
%pip install openai pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [11]:
#STEP 1: Import Libraries and Set Up API Key
import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor

load_dotenv()  # load OPENAI_API_KEY from the .env file
client = OpenAI()
MODEL = "gpt-4o-mini"

def call_openai(prompt, temperature=0.7, model=MODEL):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    return response.choices[0].message.content.strip()

def run_multiple(prompt, n=15, temperature=0.7):
    """Run the same prompt n times in parallel (for consistency testing)."""
    with ThreadPoolExecutor(max_workers=5) as ex:
        return list(ex.map(lambda _: call_openai(prompt, temperature), range(n)))

print("✅ Setup complete!" if os.getenv("OPENAI_API_KEY") else "⚠️ OPENAI_API_KEY not set")

✅ Setup complete!


## STEP 2: Create Initial Prompts
Outcome: three basic prompts that produce some output (event that is not perfect) to be used in the next step

In [12]:
#STEP 2

sentiment_prompt_v1 = """
Classify this customer message: "I love this product! It's exactly what I needed."
"""
product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""
extraction_prompt_v1 = """
Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

for name, p in [("Sentiment Analysis", sentiment_prompt_v1),
                ("Product Description", product_prompt_v1),
                ("Data Extraction", extraction_prompt_v1)]:
    print(f"--- {name} Result ---")
    print(call_openai(p), "\n")

--- Sentiment Analysis Result ---
The customer message can be classified as "Positive Feedback" or "Customer Satisfaction." 

--- Product Description Result ---
**Product Description: Wireless Precision Mouse**

Elevate your computing experience with our Wireless Precision Mouse, designed for both comfort and performance at an unbeatable price of just $29.99. Say goodbye to tangled cords and hello to seamless wireless connectivity! 

**Key Features:**

- **Ergonomic Design:** Crafted to fit comfortably in your hand, this mouse reduces strain during long hours of use, making it perfect for work or play.
  
- **Reliable Wireless Connectivity:** Enjoy a strong and stable connection with a range of up to 33 feet, thanks to advanced 2.4GHz wireless technology. 

- **Long Battery Life:** Equipped with an energy-efficient battery, our mouse provides up to 12 months of use on a single AA battery, so you can focus on what matters most without the hassle of constant replacements.

- **Adjustable

## Exploring more use cases:

### Test Cases (TechFlow Solutions: consumer tech & accessories)
The v1 prompts above are simpler: they are the **baseline**. Below is a fixed set of test inputs, from easy to tricky, that every prompt version (v1 → final) will be run against, so improvements can be compared fairly.

Prompt **templates** use the placeholder `{input}`, which is replaced by each test case.

In [13]:
#TEST CASES: same inputs for every prompt version

sentiment_cases = [
    "I love this product! It's exactly what I needed.",                                   # clear positive
    "The keyboard stopped working after a week. Very disappointed.",                       # clear negative
    "Great sound quality, but the headphones are uncomfortable after an hour.",            # mixed
    "Oh wonderful, my new charger already broke. Just what I wanted.",                     # sarcasm
    "I received the webcam today. Haven't tried it yet.",                                  # neutral
]

product_cases = [
    "Wireless mouse, $29.99",                                                              # baseline (from instructions)
    "USB-C hub, $49.99",                                                                   # very few details
    "Noise-cancelling headphones, $199, 30h battery, Bluetooth 5.3, foldable, 3 colors",   # many specs
    "LED desk lamp with wireless charging pad, $59.99",                                    # combined product
]

extraction_cases = [
    "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged.",       # baseline
    "My laptop stand arrived last Tuesday and one of the screws was missing.",                          # no order number, relative date
    "Order #88231: I bought a keyboard and a mouse. The keyboard is great, the mouse doesn't connect.", # two items, mixed feedback
    "Still waiting for my refund. Nobody answers my emails!!",                                          # almost no structured info
]

# Prompt templates: the v1 prompts from Step 2, with {input} as placeholder
sentiment_template_v1 = 'Classify this customer message: "{input}"'
product_template_v1 = "Create a product description for this product: {input}."
extraction_template_v1 = 'Extract information from this customer feedback: "{input}"'


def run_test_cases(template, cases, n=1, temperature=0.7):
    """Run a prompt template on every test case, n times each. Returns a DataFrame."""
    rows = []
    for case in cases:
        prompt = template.replace("{input}", case)
        outputs = run_multiple(prompt, n=n, temperature=temperature)
        for i, out in enumerate(outputs, start=1):
            rows.append({"input": case, "run": i, "output": out})
    return pd.DataFrame(rows)


# Quick check: run each v1 template once on all test cases
pd.set_option("display.max_colwidth", None)
display(run_test_cases(sentiment_template_v1, sentiment_cases))

,input,run,output
0,I love this product! It's exactly what I needed.,1,"The customer message can be classified as ""Positive Feedback"" or ""Customer Praise."""
1,The keyboard stopped working after a week. Very disappointed.,1,The customer message can be classified as a **product issue complaint**.
2,"Great sound quality, but the headphones are uncomfortable after an hour.",1,The customer message can be classified as **mixed feedback**. It highlights a positive aspect (great sound quality) while also mentioning a negative experience (uncomfortable after an hour).
3,"Oh wonderful, my new charger already broke. Just what I wanted.",1,This customer message can be classified as a **frustrated complaint**. The customer expresses dissatisfaction with the product they received (the charger) and uses sarcasm to convey their disappointment.
4,I received the webcam today. Haven't tried it yet.,1,"The customer message can be classified as ""Order Confirmation/Delivery Update."""


## STEP 3: Run Prompts 5 Times
Each v1 prompt is run **5 times on every test case**. All responses are saved in `results/` (CSV) so they can be compared later with the 10 and 15 run tests.

To measure consistency:
- **Sentiment**: the label in each answer is normalised (positive / negative / neutral / mixed / other). Consistency = % of runs that gave the most common label.
- **Product description & Data extraction**: long free text is never identical, so we check **format** (length, bullets, JSON, headings) and how many distinct formats appear.

In [14]:
#STEP 3: helper functions to measure consistency
import re
from collections import Counter

os.makedirs("results", exist_ok=True)

def sentiment_label(text):
    """Normalise a free-text answer to one label."""
    t = text.lower()
    if "mixed" in t:  # "mixed feedback" answers also mention positive/negative aspects
        return "mixed"
    found = [l for l in ["positive", "negative", "neutral"] if l in t]
    return found[0] if len(found) == 1 else ("multiple" if found else "other")

def output_format(text):
    """Describe the shape of an answer (used for product descriptions and extraction)."""
    t = text.strip()
    if t.startswith("{") or t.startswith("```json"):
        return "json"
    if re.search(r"^\s*[-*•]\s", t, re.M) or re.search(r"^\s*\d+\.\s", t, re.M):
        return "list"
    if "**" in t or t.startswith("#"):
        return "markdown text"
    return "plain text"

def consistency_report(df, label_fn):
    """Per input: most common label/format, % of runs that match it, distinct variants, length range."""
    df = df.assign(label=df["output"].map(label_fn), words=df["output"].str.split().str.len())
    rows = []
    for case, g in df.groupby("input", sort=False):
        top, count = Counter(g["label"]).most_common(1)[0]
        rows.append({
            "input": case,
            "most_common": top,
            "consistency_%": round(100 * count / len(g)),
            "distinct_variants": g["label"].nunique(),
            "unique_texts": g["output"].nunique(),
            "words_min-max": f'{g["words"].min()}-{g["words"].max()}',
        })
    return pd.DataFrame(rows)

In [15]:
#STEP 3: run each v1 prompt 5 times on all test cases
N_RUNS = 5

sentiment_v1_5 = run_test_cases(sentiment_template_v1, sentiment_cases, n=N_RUNS)
product_v1_5 = run_test_cases(product_template_v1, product_cases, n=N_RUNS)
extraction_v1_5 = run_test_cases(extraction_template_v1, extraction_cases, n=N_RUNS)

# Record all responses
sentiment_v1_5.to_csv("results/step3_sentiment_v1_5runs.csv", index=False)
product_v1_5.to_csv("results/step3_product_v1_5runs.csv", index=False)
extraction_v1_5.to_csv("results/step3_extraction_v1_5runs.csv", index=False)

print("=== Sentiment Analysis: 5 runs ===")
display(consistency_report(sentiment_v1_5, sentiment_label))
print("=== Product Description: 5 runs ===")
display(consistency_report(product_v1_5, output_format))
print("=== Data Extraction: 5 runs ===")
display(consistency_report(extraction_v1_5, output_format))

=== Sentiment Analysis: 5 runs ===


,input,most_common,consistency_%,distinct_variants,unique_texts,words_min-max
0,I love this product! It's exactly what I needed.,positive,100,1,5,9-13
1,The keyboard stopped working after a week. Very disappointed.,other,60,2,5,12-28
2,"Great sound quality, but the headphones are uncomfortable after an hour.",mixed,100,1,4,26-29
3,"Oh wonderful, my new charger already broke. Just what I wanted.",negative,60,2,5,22-29
4,I received the webcam today. Haven't tried it yet.,other,100,1,5,11-30


=== Product Description: 5 runs ===


,input,most_common,consistency_%,distinct_variants,unique_texts,words_min-max
0,"Wireless mouse, $29.99",list,100,1,5,235-303
1,"USB-C hub, $49.99",list,100,1,5,248-325
2,"Noise-cancelling headphones, $199, 30h battery, Bluetooth 5.3, foldable, 3 colors",list,100,1,5,244-278
3,"LED desk lamp with wireless charging pad, $59.99",list,100,1,5,251-298


=== Data Extraction: 5 runs ===


,input,most_common,consistency_%,distinct_variants,unique_texts,words_min-max
0,I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged.,list,100,1,5,25-29
1,My laptop stand arrived last Tuesday and one of the screws was missing.,list,80,2,5,15-44
2,"Order #88231: I bought a keyboard and a mouse. The keyboard is great, the mouse doesn't connect.",plain text,60,2,5,18-36
3,Still waiting for my refund. Nobody answers my emails!!,list,80,2,5,24-56


In [16]:
#STEP 3: look at the raw answers (change the DataFrame / input to inspect others)
display(sentiment_v1_5[["input", "run", "output"]])

,input,run,output
0,I love this product! It's exactly what I needed.,1,The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
1,I love this product! It's exactly what I needed.,2,The customer message can be classified as positive feedback or a positive review.
2,I love this product! It's exactly what I needed.,3,This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
3,I love this product! It's exactly what I needed.,4,This customer message can be classified as Positive Feedback or Positive Customer Experience.
4,I love this product! It's exactly what I needed.,5,This customer message can be classified as **Positive Feedback**.
5,The keyboard stopped working after a week. Very disappointed.,1,This customer message can be classified as a **complaint**. The customer is expressing dissatisfaction with a product (the keyboard) due to it malfunctioning shortly after purchase.
6,The keyboard stopped working after a week. Very disappointed.,2,The customer message can be classified as a **complaint**. The customer expresses dissatisfaction with the product's performance (the keyboard) and indicates disappointment with their experience.
7,The keyboard stopped working after a week. Very disappointed.,3,"This customer message can be classified as a **product issue/complaint**. The customer is expressing dissatisfaction with the performance of the keyboard, indicating a problem with the product quality."
8,The keyboard stopped working after a week. Very disappointed.,4,"This customer message can be classified as a **complaint**. The customer expresses dissatisfaction with the product's performance and indicates disappointment, suggesting a negative experience."
9,The keyboard stopped working after a week. Very disappointed.,5,The customer message can be classified as a complaint or negative feedback.


### Step 3 Observations (5 runs)
Based on `results/step3_*_v1_5runs.csv` (5 runs × every test case) and the manual runs in `results/step3_manual_5runs.md`.

**Are responses consistent in format?**
- **Sentiment:** No. Every answer is a full sentence ("The customer message can be classified as…") instead of a single label. Sometimes in bold, sometimes in quotes, often with an explanation added (9–33 words).
- **Product description:** Partly. Always a markdown text with a bulleted feature list, but the header changes between runs ("Product Name:", "Product Description: … – $29.99", "Product Title:", a marketing headline), and length varies from ~220 to ~330 words.
- **Data extraction:** No. Sometimes a bullet list with an intro line ("Here is the extracted information…"), sometimes plain paragraphs ("The customer feedback indicates…"). Never JSON or any fixed structure.

**Are responses consistent in content?**
- **Sentiment:** The general meaning is usually right, but the *labels* are not the ones the task requires:

  | Message | Labels over 5 runs |
  |---|---|
  | Clear positive | positive ×5 (often two labels: "Positive Feedback **or** Customer Satisfaction") |
  | Clear negative | "complaint" ×4, negative ×1 |
  | Mixed | "mixed feedback" ×5 (not one of positive/negative/neutral) |
  | Sarcasm | negative ×2, "frustration/complaint" ×3 |
  | Neutral | invented category ×5 ("Order Update", "Product Receipt Confirmation", …) |

- **Product description:** No. The model invents a different product name (Sleek Wireless Mouse, SerenitySound, MultiPort USB-C Hub…) and **invents specs that were never given and contradict each other** (mouse battery AA vs AAA, 6 vs 12 months; DPI settings only in some runs). Given specs (e.g. "30h battery") are sometimes missing.
- **Data extraction:** No. Field names change from run to run ("Packaging Condition" vs "Issue"; "Items Purchased" vs "Purchased Items" vs "Products Purchased"), the order number appears as `#12345` or `12345`, and for the harder cases (no order number, refund complaint) the model writes a summary instead of extracting fields.

**What types of variations do you see?**
1. **Label variation**: synonyms and invented categories instead of a fixed label set.
2. **Multiple answers**: "X or Y" instead of one decision.
3. **Format variation**: sentence vs label, list vs paragraph, different headers, extra explanations.
4. **Hallucination**: invented product names and specs in product descriptions.
5. **Schema variation**: different field names, value formats, and missing fields in data extraction.

**Root cause:** the v1 prompts don't define the allowed labels, the output format, length, or which fields to extract, so the model fills the gaps differently on each run. **Hard cases** (mixed, sarcasm, neutral, missing info) produce the most variation; the easy baseline examples look almost consistent.

## STEP 4: Run Prompts 10 Times
**What to do**:

- Run each prompt 10 times
- Compare results with the 5-run test
- Identify if more runs reveal additional failure patterns

**Expected outcome**: You notice that with more runs, consistency may decrease or new failure patterns emerge.

In [17]:
# STEP 4: run each v1 prompt 10 times on all test cases
N_RUNS = 10

sentiment_v1_10 = run_test_cases(sentiment_template_v1, sentiment_cases, n=N_RUNS)
product_v1_10 = run_test_cases(product_template_v1, product_cases, n=N_RUNS)
extraction_v1_10 = run_test_cases(extraction_template_v1, extraction_cases, n=N_RUNS)

# Record all responses
sentiment_v1_10.to_csv("results/step4_sentiment_v1_10runs.csv", index=False)
product_v1_10.to_csv("results/step4_product_v1_10runs.csv", index=False)
extraction_v1_10.to_csv("results/step4_extraction_v1_10runs.csv", index=False)
print("✅ 10-run results saved in results/")

✅ 10-run results saved in results/


In [18]:
# STEP 4: compare the 10-run test with the 5-run test
# Keep this cell self-contained so it also works after a kernel restart.
import re
import pandas as pd
from collections import Counter

def sentiment_label(text):
    t = text.lower()
    if "mixed" in t:
        return "mixed"
    found = [label for label in ["positive", "negative", "neutral"] if label in t]
    return found[0] if len(found) == 1 else ("multiple" if found else "other")

def output_format(text):
    t = text.strip()
    if t.startswith("{") or t.startswith("```json"):
        return "json"
    if re.search(r"^\s*[-*•]\s", t, re.M) or re.search(r"^\s*\d+\.\s", t, re.M):
        return "list"
    if "**" in t or t.startswith("#"):
        return "markdown text"
    return "plain text"

def consistency_report(df, label_fn):
    df = df.assign(label=df["output"].map(label_fn), words=df["output"].str.split().str.len())
    rows = []
    for case, group in df.groupby("input", sort=False):
        top, count = Counter(group["label"]).most_common(1)[0]
        rows.append({
            "input": case,
            "most_common": top,
            "consistency_%": round(100 * count / len(group)),
            "distinct_variants": group["label"].nunique(),
            "unique_texts": group["output"].nunique(),
            "words_min-max": f'{group["words"].min()}-{group["words"].max()}',
        })
    return pd.DataFrame(rows)


sentiment_v1_5 = pd.read_csv("results/step3_sentiment_v1_5runs.csv")
product_v1_5 = pd.read_csv("results/step3_product_v1_5runs.csv")
extraction_v1_5 = pd.read_csv("results/step3_extraction_v1_5runs.csv")
sentiment_v1_10 = pd.read_csv("results/step4_sentiment_v1_10runs.csv")
product_v1_10 = pd.read_csv("results/step4_product_v1_10runs.csv")
extraction_v1_10 = pd.read_csv("results/step4_extraction_v1_10runs.csv")

def classified_as(text):
    """The category the model chose, e.g. 'complaint', 'order update' (text after 'classified as')."""
    m = re.search(r"classified as (?:an? )?(.+?)(?:\.|$)", text, re.I | re.M)
    return re.sub(r'[*"“”]', "", m.group(1)).strip().lower() if m else "(no 'classified as')"

def header_style(text):
    """How a product description starts, e.g. 'Product Name:', 'Product Description:', or a headline."""
    first = re.sub(r"[*#]", "", text.strip().split("\n")[0]).strip()
    return first.split(":")[0] + ":" if ":" in first[:25] else "headline"

def field_names(text):
    """Bold field names in an extraction answer, e.g. 'Order Number', 'Issue'."""
    names = re.findall(r"\*\*([^*:]+?):?\*\*", text)
    return ", ".join(names) if names else "(no fields: paragraph)"

def compare_runs(df5, df10, label_fn, variant_fn):
    """Consistency at 5 vs 10 runs, plus variants that only appeared in the 10-run test."""
    r5 = consistency_report(df5, label_fn).set_index("input")
    r10 = consistency_report(df10, label_fn).set_index("input")
    rows = []
    for case in r10.index:
        v5 = set(df5.loc[df5["input"] == case, "output"].map(variant_fn))
        v10 = set(df10.loc[df10["input"] == case, "output"].map(variant_fn))
        rows.append({
            "input": case,
            "consistency_% (5)": r5.loc[case, "consistency_%"],
            "consistency_% (10)": r10.loc[case, "consistency_%"],
            "variants (5)": len(v5),
            "variants (10)": len(v10),
            "NEW in 10 runs": "; ".join(sorted(v10 - v5)) or "-",
            "words (10)": r10.loc[case, "words_min-max"],
        })
    return pd.DataFrame(rows)

print("=== Sentiment Analysis: 5 vs 10 runs (variants = category named after 'classified as') ===")
display(compare_runs(sentiment_v1_5, sentiment_v1_10, sentiment_label, classified_as))
print("=== Product Description: 5 vs 10 runs (variants = header style) ===")
display(compare_runs(product_v1_5, product_v1_10, output_format, header_style))
print("=== Data Extraction: 5 vs 10 runs (variants = set of field names) ===")
display(compare_runs(extraction_v1_5, extraction_v1_10, output_format, field_names))

=== Sentiment Analysis: 5 vs 10 runs (variants = category named after 'classified as') ===


,input,consistency_% (5),consistency_% (10),variants (5),variants (10),NEW in 10 runs,words (10)
0,I love this product! It's exactly what I needed.,100,100,4,3,positive feedback or a review,9-12
1,The keyboard stopped working after a week. Very disappointed.,60,90,3,7,product complaint or customer dissatisfaction; product issue complaint; product issue or defective product complaint; product issue/defect complaint,11-30
2,"Great sound quality, but the headphones are uncomfortable after an hour.",100,100,1,3,mixed feedback comment; product review with a mixed sentiment,25-31
3,"Oh wonderful, my new charger already broke. Just what I wanted.",60,60,5,5,complaint; frustration or disappointment,21-32
4,I received the webcam today. Haven't tried it yet.,100,100,3,5,order update or product receipt acknowledgment; order update or product received inquiry; order update/confirmation,9-33


=== Product Description: 5 vs 10 runs (variants = header style) ===


,input,consistency_% (5),consistency_% (10),variants (5),variants (10),NEW in 10 runs,words (10)
0,"Wireless mouse, $29.99",100,100,2,2,-,240-291
1,"USB-C hub, $49.99",100,100,2,3,Product Title:,269-360
2,"Noise-cancelling headphones, $199, 30h battery, Bluetooth 5.3, foldable, 3 colors",100,100,1,3,Product Name:; Product Title:,220-297
3,"LED desk lamp with wireless charging pad, $59.99",100,100,2,2,-,235-315


=== Data Extraction: 5 vs 10 runs (variants = set of field names) ===


,input,consistency_% (5),consistency_% (10),variants (5),variants (10),NEW in 10 runs,words (10)
0,I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged.,100,100,3,4,"Item Number, Order Date, Delivery Speed, Packaging Condition",25-29
1,My laptop stand arrived last Tuesday and one of the screws was missing.,80,70,4,2,-,17-37
2,"Order #88231: I bought a keyboard and a mouse. The keyboard is great, the mouse doesn't connect.",60,90,3,5,"Order Number, Items Purchased; Order Number, Items Purchased, Feedback; Order Number, Items Purchased, Feedback on Keyboard, Feedback on Mouse; Order Number, Purchased Items, Feedback on Keyboard, Feedback on Mouse",18-56
3,Still waiting for my refund. Nobody answers my emails!!,80,80,4,5,"Issue, Concern, Emotion; Issue, Response, Sentiment; Refund Status, Lack of Communication",21-71


### Step 4 Observations (10 runs vs 5 runs)
Based on `results/step4_*_v1_10runs.csv` compared with `results/step3_*_v1_5runs.csv`.

**Did consistency go down with more runs?**
The headline "consistency %" stayed almost the same (only sarcasm moved, 60% → 70%), but that number is misleading: it only checks the broad label/format. The number of **distinct variants roughly doubled** for most inputs, so the real consistency went down.

- **Sentiment:** Variants went up on 4 of 5 messages: clear negative 3 → 6, sarcasm 2 → 4, neutral 4 → 7, clear positive 2 → 3. The neutral message never got the same category twice in a row: 7 different invented names in 10 runs ("Product Receipt Confirmation", "Order Update/Confirmation", "Order Confirmation/Delivery Update"…). Only the mixed message stayed stable ("mixed feedback" ×9, "mixed review" ×1), but "mixed" is still not one of the allowed labels.
- **Product description:** The format check stays at 100% "list" and the dominant header became *more* common ("Product Description:" 9/10 for mouse, 10/10 for USB-C hub). But new header styles appeared ("Product Title:", marketing headlines like "Illuminate Your Space/Workspace…" in 5/10 lamp runs), invented brand names are still there ("Sleek Wireless Mouse", "Serenity Sound"), and length still ranges ~200–330 words. Given specs (30h, Bluetooth 5.3, foldable, colors) were included in 10/10 headphone runs, so this task is more consistent in *content* than in *style*.
- **Data extraction:** Field names drifted the most. Variants went up on every input: baseline 2 → 3, laptop stand 2 → 4, two-item order 5 → 7, refund complaint 5 → 8. The refund complaint produced 8 different field sets in 10 runs, and it is the only input whose consistency % dropped (100 → 90%) because some runs gave a plain paragraph with no fields at all.

**New failure patterns that only appeared in 10 runs** (see the "NEW in 10 runs" column):
- **Even longer, hedged labels:** "complaint or negative feedback regarding product quality and performance", "complaint or feedback". The model keeps adding "X or Y" and qualifiers instead of choosing one label.
- **Sarcasm read as soft emotion:** new labels "frustration or dissatisfaction" and "complaint or feedback". It sees the anger but never gives a clean "negative".
- **Wrong field meaning:** for the laptop stand, one run put the *arrival* date under **"Order Date"** ("Order Date: Arrived last Tuesday"). The value is in the wrong field, which is worse than a different name.
- **Case and synonym drift in field names:** "Delivery Date" vs "Delivery date", "Delivery Speed" vs "Delivery", "Items Purchased" vs "Purchased Items", "Feedback on Keyboard" vs "Keyboard Feedback". A program reading these keys would break.
- **Structure changes for multiple items:** for order #88231 some runs nest the feedback inside the item list ("Keyboard (feedback: great)") instead of a separate feedback field, so the same information is in a different place.
- **Interpretation instead of extraction:** refund complaint runs added "Frustration" and "Urgency" as fields. These are the model's opinions, not information found in the text.
- **Relative dates never resolved:** "last Tuesday" is copied as-is (sometimes with a note like "specific date not provided"), never turned into a date or flagged as unknown in a standard way.

**Conclusion:** 5 runs were not enough to see the full problem. With 10 runs the main pattern stays the same but the **long tail of variants grows**: each extra run is a new chance for a new label, header or field name. Hard inputs (neutral, sarcasm, missing order number, refund complaint) get worse the most. The cause is the same as in Step 3: no fixed label set, no output format, no field schema. So the fix must **constrain the output** (allowed labels only, a JSON schema with fixed keys and `null` for missing values, fixed length/structure for descriptions, no invented specs) and probably **lower the temperature**. Adding more runs only shows more variation; it does not make the output more stable.

## STEP 5: Run Prompts 15 Times and Create Failure Analysis 

**What to do:**
1. Run each prompt 15 times
2. Create a comprehensive failure analysis table
3. Document **specific** failure patterns for each prompt

In [19]:
#STEP 5: run each v1 prompt 15 times on all test cases
# Re-uses the saved CSVs if they already exist, so re-running the notebook does not repeat the API calls.
# Delete the step5_*.csv files to generate a fresh 15-run test.
N_RUNS = 15

def run_or_load(name, template, cases):
    path = f"results/step5_{name}_v1_15runs.csv"
    if os.path.exists(path):
        return pd.read_csv(path)
    df = run_test_cases(template, cases, n=N_RUNS)
    df.to_csv(path, index=False)
    return df

sentiment_v1_15 = run_or_load("sentiment", sentiment_template_v1, sentiment_cases)
product_v1_15 = run_or_load("product", product_template_v1, product_cases)
extraction_v1_15 = run_or_load("extraction", extraction_template_v1, extraction_cases)

print(f"✅ 15-run results in results/: {len(sentiment_v1_15)} sentiment, "
      f"{len(product_v1_15)} product, {len(extraction_v1_15)} extraction responses")

✅ 15-run results in results/: 75 sentiment, 60 product, 60 extraction responses


In [20]:
# STEP 5: failure analysis (uses the helper functions from the Step 4 cell)
import json

sentiment_v1_15 = pd.read_csv("results/step5_sentiment_v1_15runs.csv")
product_v1_15 = pd.read_csv("results/step5_product_v1_15runs.csv")
extraction_v1_15 = pd.read_csv("results/step5_extraction_v1_15runs.csv")

def field_labels(text):
    """Field names in an extraction answer, bold or not (e.g. '- Order Number: ...').
    Skips an intro/title line such as 'Here is the extracted information:'; field names are 1-3 words."""
    lines = text.strip().split("\n")
    if lines[0].rstrip("*: ").strip() and lines[0].rstrip("* ").endswith(":"):
        lines = lines[1:]
    names = re.findall(r"^\s*(?:[-*•]|\d+\.)?\s*\**([A-Z][A-Za-z/]*(?: [A-Za-z/]+){0,2})\**\s*:", "\n".join(lines), re.M)
    return ", ".join(names) if names else "(no fields: paragraph)"

# Facts the model adds that were NOT in the product input (regex, matched on whole words)
INVENTED_SPECS = {
    "Wireless mouse, $29.99": r"\bdpi\b|2\.4 ?ghz|\d+ ?(?:feet|ft)\b|\baaa?\b",
    "USB-C hub, $49.99": r"\b4k\b|\bhdmi\b|\bgbps\b|\bsd card|\bethernet\b|usb 3",
    "Noise-cancelling headphones, $199, 30h battery, Bluetooth 5.3, foldable, 3 colors": r"quick charg|fast charg|\bmicrophone\b|\bmic\b|\bapp\b",
    "LED desk lamp with wireless charging pad, $59.99": r"\bqi\b|color temperature|brightness levels|usb port",
}

def is_valid(task, case, text):
    """Does the answer meet what the task actually needs?"""
    t = text.strip().lower()
    if task == "sentiment":   # exactly one allowed label, nothing else
        return t.strip(".*\"' ") in {"positive", "negative", "neutral"}
    if task == "product":     # no facts that were not given
        return not re.search(INVENTED_SPECS[case], t)
    if task == "extraction":  # machine-readable JSON
        try:
            json.loads(re.sub(r"^```(?:json)?|```$", "", text.strip()).strip())
            return True
        except ValueError:
            return False

VALID_RULE = {"sentiment": "single label (positive/negative/neutral)",
              "product": "no invented specs",
              "extraction": "valid JSON"}

def failure_table(task, runs, variant_fn):
    """One row per test case: strict consistency at 5/10/15 runs, unique answers, validity."""
    rows = []
    for case, g15 in runs[15].groupby("input", sort=False):
        row = {"task": task, "input": case[:60]}
        for n, df in runs.items():
            v = df.loc[df["input"] == case, "output"].map(variant_fn)
            row[f"consistency_% ({n})"] = round(100 * v.value_counts().iloc[0] / len(v))
        v15 = g15["output"].map(variant_fn)
        row["variants (15)"] = v15.nunique()
        row["unique_answers (15)"] = g15["output"].nunique()
        row["most_common (15)"] = v15.value_counts().index[0]
        row["words"] = f'{g15["output"].str.split().str.len().min()}-{g15["output"].str.split().str.len().max()}'
        row[f"valid_%"] = round(100 * g15["output"].map(lambda o: is_valid(task, case, o)).mean())
        rows.append(row)
    return pd.DataFrame(rows)

failure_analysis = pd.concat([
    failure_table("sentiment", {5: sentiment_v1_5, 10: sentiment_v1_10, 15: sentiment_v1_15}, classified_as),
    failure_table("product", {5: product_v1_5, 10: product_v1_10, 15: product_v1_15}, header_style),
    failure_table("extraction", {5: extraction_v1_5, 10: extraction_v1_10, 15: extraction_v1_15}, field_labels),
], ignore_index=True)
failure_analysis.to_csv("results/step5_failure_analysis.csv", index=False)

print("Consistency = % of runs that gave the most common category (sentiment), header style (product) or field set (extraction).")
print("valid_% rules:", VALID_RULE)
display(failure_analysis)

# Overall per prompt: target is > 80% consistency at 15 runs on every input
summary = failure_analysis.groupby("task", sort=False).agg(
    avg_consistency_15=("consistency_% (15)", "mean"),
    inputs_above_80=("consistency_% (15)", lambda s: f"{(s > 80).sum()}/{len(s)}"),
    avg_valid=("valid_%", "mean"),
).round()
display(summary)

Consistency = % of runs that gave the most common category (sentiment), header style (product) or field set (extraction).
valid_% rules: {'sentiment': 'single label (positive/negative/neutral)', 'product': 'no invented specs', 'extraction': 'valid JSON'}


,task,input,consistency_% (5),consistency_% (10),consistency_% (15),variants (15),unique_answers (15),most_common (15),words,valid_%
0,sentiment,I love this product! It's exactly what I needed.,40,60,53,3,9,positive feedback or customer satisfaction,9-12,0
1,sentiment,The keyboard stopped working after a week. Very disappointed,60,40,67,4,9,complaint,9-35,0
2,sentiment,"Great sound quality, but the headphones are uncomfortable af",100,80,93,2,13,mixed feedback,23-28,0
3,sentiment,"Oh wonderful, my new charger already broke. Just what I want",20,40,40,8,12,complaint or negative feedback,17-24,0
4,sentiment,I received the webcam today. Haven't tried it yet.,40,50,47,6,13,product receipt confirmation,10-32,0
5,product,"Wireless mouse, $29.99",80,80,93,2,15,Product Description:,228-286,47
6,product,"USB-C hub, $49.99",80,70,47,3,15,Product Description:,254-347,0
7,product,"Noise-cancelling headphones, $199, 30h battery, Bluetooth 5.",100,80,87,2,15,Product Description:,205-286,93
8,product,"LED desk lamp with wireless charging pad, $59.99",80,60,60,2,15,headline,221-304,0
9,extraction,I ordered item #12345 on March 15th. The delivery was fast b,60,60,60,7,11,"Order Number, Order Date, Delivery Speed, Packaging Condition",25-27,0


,avg_consistency_15,inputs_above_80,avg_valid
task,,,
sentiment,60.0,1/5,0.0
product,72.0,2/4,35.0
extraction,46.0,0/4,0.0


### Step 5: Failure Analysis (15 runs)
Based on `results/step5_*_v1_15runs.csv` (195 responses) and the table above, which is also saved as `results/step5_failure_analysis.csv`.

**How it is measured** (stricter than Step 3, which only checked the broad label/format):
- **Consistency %**: share of the 15 runs that gave the *most common* category (sentiment), header style (product), or set of field names (extraction). **Target: > 80%.**
- **Valid %**: share of runs that give what the task actually needs: one of `positive / negative / neutral` (sentiment), no specs that were not in the input (product), parseable JSON (extraction).

#### Failure analysis table

| Prompt | Avg consistency (15) | Inputs > 80% | Unique answers (per input, of 15) | Valid % | Verdict |
|---|---|---|---|---|---|
| Sentiment | 60% | 1/5 (only "mixed") | 9–13 | **0%** | ❌ Fails |
| Product description | 72% | 2/4 | 15 (every answer different) | 35% | ❌ Fails |
| Data extraction | 46% | 0/4 | 11–15 | **0%** | ❌ Fails |

**None of the three v1 prompts meets the target.** No run of the sentiment or extraction prompt gave a usable answer. The only sentiment input above 80% ("mixed feedback", 93%) is consistent, but it uses a label the task does not allow.

#### Sentiment analysis: specific failure patterns
| # | Pattern | Evidence (15 runs) |
|---|---|---|
| S1 | **Sentence instead of a label** | 75/75 answers are a sentence ("This customer message can be classified as…"), 9–35 words. 0% valid. |
| S2 | **Invented categories** | Neutral webcam message: never "neutral". 6 different made-up names ("Product Receipt Confirmation", "Order Fulfillment Update", "Order Update or Product Receipt Acknowledgment"…). Negative keyboard message: "complaint" ×10 instead of "negative". |
| S3 | **Hedging "X or Y"** | Clear positive: "positive feedback **or** customer satisfaction" 8/15, so it is never a single decision. |
| S4 | **Sarcasm unstable** | "Oh wonderful… already broke": 8 variants in 15 runs, the worst of all inputs. It is called "negative" in only 7/15 runs; the rest say "frustration/complaint", "frustrated complaint"… Consistency fell 80% → 40% → 40% (5 → 10 → 15 runs). |
| S5 | **"Mixed" outside the label set** | Mixed headphones message: "mixed feedback" 14/15. Stable, but not allowed: the prompt never says what to do with mixed messages. |

#### Product description: specific failure patterns
| # | Pattern | Evidence (15 runs) |
|---|---|---|
| P1 | **Hallucinated specs** | USB-C hub: 15/15 runs invent ports (4K HDMI, SD card, USB 3.0, 5 Gbps), so 0% valid. Lamp: 15/15 invent brightness levels / color temperature / Qi. Mouse: 8/15 invent a "33 feet" range, 2.4 GHz, AA battery. Only the headphones (input with full specs) stay mostly factual (93%). |
| P2 | **Invented product names** | "UltraConnect USB-C Hub" (5 runs), "MultiPort USB-C Hub", "Versatile USB-C Hub", "SerenitySound Headphones", "Sleek Wireless Mouse". The same product gets a different brand name. |
| P3 | **Header style changes** | USB-C hub: "Product Description:" 7, "Product Name:" 6, "Product Title:" 2 (47% consistency). Lamp: marketing headline ("Illuminate Your Workspace…") 9/15 vs "Product Description:" 6/15. |
| P4 | **Length not controlled** | 205–347 words; the USB-C hub alone varies by ~90 words. No length was asked for. |
| P5 | **Every text is unique** | 60/60 answers are different texts. Normal for creative writing, so structure (header, sections, length) must be fixed to make it consistent. |

**Fewer input details means more hallucination**: the less the input says, the more the model invents (hub/lamp 0% valid vs headphones 93%).

#### Data extraction: specific failure patterns
| # | Pattern | Evidence (15 runs) |
|---|---|---|
| E1 | **No machine-readable format** | 0/60 answers are JSON. Always a bullet list or paragraph, usually with an intro line ("Here is the extracted information…", "Customer Feedback Summary:"). |
| E2 | **Field names drift** | Baseline order: 7 different field sets ("Order Number" / "Order Item Number" / "Order Item" / "Item Ordered"; "Delivery Speed" / "Delivery" / "Delivery Experience"; "Packaging Condition" / "Issue"). |
| E3 | **Summary instead of fields** | Refund complaint: 5/15 runs have no field names at all, only sentences. Laptop stand: 5/15 the same. |
| E4 | **Interpretation added as data** | Refund complaint gets fields like "Sentiment", "Frustration", "Action needed", "Concern": the model's opinion, not information in the text. 9 different field sets in 15 runs (33% consistency, the worst input). |
| E5 | **Value format varies** | Order number written `#12345` in 12 runs and `12345` in 3. "Last Tuesday" is never turned into a date or marked as missing. |
| E6 | **Structure changes for multiple items** | Order #88231: feedback is sometimes its own field per item ("Feedback on Keyboard"), sometimes nested inside the item list, sometimes one combined "Feedback" line. |

#### Trend 5 → 10 → 15 runs
- More runs did **not** make the results converge. Sarcasm (80 → 40 → 40%), laptop stand (80 → 30 → 40%) and USB-C hub (60 → 100 → 47%) show that a small sample can look good **by chance**.
- The number of unique answers grows with almost every run: the model rarely repeats itself word for word at temperature 0.7.
- **Hard inputs fail first and worst**: sarcasm, neutral, missing order number and the refund complaint are the lowest in every test.

#### What the next iterations must fix
1. **Sentiment (Step 6)**: give the allowed labels, say "answer with one word only", and define how to handle mixed and sarcastic messages.
2. **Product description (Step 7)**: fixed structure (title, short paragraph, N bullets), a word limit, a style guide, and "use only the facts given; do not invent specs or brand names".
3. **Data extraction (Step 8)**: a JSON schema with fixed keys, a value format (e.g. order number without `#`), `null` for missing information, and "extract only, do not interpret".
4. **All prompts**: lower the temperature for the classification and extraction tasks, and re-test with the same 15-run setup.

## Step 6: Improve Sentiment Analysis Prompt

**What to do:**
1. Rewrite the sentiment analysis prompt with:
   - Clearer instructions
   - Explicit output format requirement
   - Constraints (single word response)

In [21]:
#STEP 6: improve the sentiment prompt (v2) to fix the failures found in Step 5
# Fixes: S1 sentence instead of label -> one-word output | S2 invented categories -> fixed label set
#        S3 "X or Y" hedging -> exactly one label    | S4 sarcasm -> judge the real meaning
#        S5 "mixed" outside the set -> explicit rule for mixed messages
import os
import pandas as pd

# This cell uses helpers from earlier steps. After a kernel restart, run all cells above first
# (VS Code: "Run Above" / Jupyter: Run > Run All Above Selected Cell).
_needed = ["sentiment_cases", "run_test_cases", "classified_as", "is_valid"]
_missing = [n for n in _needed if n not in globals()]
if _missing:
    raise RuntimeError(f"Run the cells above first (Steps 1-5). Missing: {', '.join(_missing)}")
sentiment_v1_15 = pd.read_csv("results/step5_sentiment_v1_15runs.csv")

sentiment_template_v2 = """You are a sentiment classifier for TechFlow Solutions customer messages.

Classify the sentiment of the customer message below.

Allowed labels (use exactly one):
- positive: the customer is satisfied or happy with the product or service
- negative: the customer is unhappy, complains, or reports a problem
- neutral: the customer states facts or asks something without expressing an opinion

Rules:
- If the message is sarcastic, classify what the customer really means, not the literal words.
- If the message contains both praise and a complaint, answer neutral.

Output format: reply with only the label in lowercase, as a single word. No punctuation, no explanation.

Customer message: "{input}"
"""

# Expected answer for each test case (used to check accuracy, not only consistency)
sentiment_expected = {
    "I love this product! It's exactly what I needed.": "positive",
    "The keyboard stopped working after a week. Very disappointed.": "negative",
    "Great sound quality, but the headphones are uncomfortable after an hour.": "neutral",   # mixed -> neutral (rule)
    "Oh wonderful, my new charger already broke. Just what I wanted.": "negative",           # sarcasm
    "I received the webcam today. Haven't tried it yet.": "neutral",
}

# Same setup as v1: 15 runs per test case, temperature 0.7 (only the prompt changes)
path = "results/step6_sentiment_v2_15runs.csv"
if os.path.exists(path):   # re-use saved runs; delete the file to run again
    sentiment_v2_15 = pd.read_csv(path)
else:
    sentiment_v2_15 = run_test_cases(sentiment_template_v2, sentiment_cases, n=15, temperature=0.7)
    sentiment_v2_15.to_csv(path, index=False)

def sentiment_v1_vs_v2(v1, v2):
    """Per test case: exact-answer consistency, validity and accuracy for v1 and v2."""
    rows = []
    for case, g2 in v2.groupby("input", sort=False):
        g1 = v1[v1["input"] == case]
        a2 = g2["output"].str.strip().str.lower()
        rows.append({
            "input": case[:60],
            "expected": sentiment_expected[case],
            "v1 consistency_%": round(100 * g1["output"].map(classified_as).value_counts().iloc[0] / len(g1)),
            "v2 consistency_%": round(100 * a2.value_counts().iloc[0] / len(a2)),
            "v1 unique answers": g1["output"].nunique(),
            "v2 unique answers": a2.nunique(),
            "v2 answers": dict(a2.value_counts()),
            "v1 valid_%": round(100 * g1["output"].map(lambda o: is_valid("sentiment", case, o)).mean()),
            "v2 valid_%": round(100 * a2.isin(["positive", "negative", "neutral"]).mean()),
            "v2 accuracy_%": round(100 * (a2 == sentiment_expected[case]).mean()),
        })
    return pd.DataFrame(rows)

sentiment_comparison = sentiment_v1_vs_v2(sentiment_v1_15, sentiment_v2_15)
sentiment_comparison.to_csv("results/step6_sentiment_v1_vs_v2.csv", index=False)
display(sentiment_comparison)
print(f"Average consistency: v1 {sentiment_comparison['v1 consistency_%'].mean():.0f}% -> v2 {sentiment_comparison['v2 consistency_%'].mean():.0f}%")
print(f"Inputs above 80%: v1 {(sentiment_comparison['v1 consistency_%'] > 80).sum()}/5 -> v2 {(sentiment_comparison['v2 consistency_%'] > 80).sum()}/5")
print(f"Average accuracy v2: {sentiment_comparison['v2 accuracy_%'].mean():.0f}%")

,input,expected,v1 consistency_%,v2 consistency_%,v1 unique answers,v2 unique answers,v2 answers,v1 valid_%,v2 valid_%,v2 accuracy_%
0,I love this product! It's exactly what I needed.,positive,53,100,9,1,{'positive': 15},0,100,100
1,The keyboard stopped working after a week. Very disappointed,negative,67,100,9,1,{'negative': 15},0,100,100
2,"Great sound quality, but the headphones are uncomfortable af",neutral,93,100,13,1,{'neutral': 15},0,100,100
3,"Oh wonderful, my new charger already broke. Just what I want",negative,40,100,12,1,{'negative': 15},0,100,100
4,I received the webcam today. Haven't tried it yet.,neutral,47,100,13,1,{'neutral': 15},0,100,100


Average consistency: v1 60% -> v2 100%
Inputs above 80%: v1 1/5 -> v2 5/5
Average accuracy v2: 100%


### Step 6 Observations: sentiment v1 vs v2 (15 runs each)
Based on `results/step6_sentiment_v2_15runs.csv`; the comparison table is saved as `results/step6_sentiment_v1_vs_v2.csv`. Same 5 test messages, same model (`gpt-4o-mini`) and same temperature (0.7) as v1, so **only the prompt changed**.

**What changed in the prompt** (each change targets a failure from Step 5):

| Step 5 failure | v2 fix |
|---|---|
| S1 Sentence instead of a label | "Reply with only the label in lowercase, as a single word. No punctuation, no explanation." |
| S2 Invented categories ("complaint", "Product Receipt Confirmation") | Fixed list of 3 allowed labels, each with a short definition |
| S3 Hedging "X or Y" | "Use exactly one" label |
| S4 Sarcasm unstable | "Classify what the customer really means, not the literal words." |
| S5 "Mixed" outside the label set | Explicit rule: praise + complaint → `neutral` |
| (general) No context | Role: "sentiment classifier for TechFlow Solutions customer messages" |

**Results**

| Metric | v1 | v2 |
|---|---|---|
| Average consistency | 60% | **100%** |
| Inputs above 80% | 1/5 | **5/5** |
| Unique answers per input (of 15) | 9–13 | **1** |
| Valid answers (single allowed label) | 0% | **100%** |
| Accuracy vs expected label | n/a (never valid) | **100%** |

- Every run gave the same single word for every message (75/75 answers valid and correct).
- **The hardest cases were fixed by the rules**: sarcasm went from 40% to 100% (`negative` ×15), and the neutral webcam message from 6 invented categories to `neutral` ×15.
- The **definitions** of each label did as much work as the format constraint: "reports a problem" → negative, "states facts without opinion" → neutral. These remove the reasons the model had for inventing categories.

**Limitations: why this is not the end of sentiment work**
- **Only 5 test messages.** 100% here shows the prompt is consistent on these cases, not that it is correct on all customer messages. Step 9 (few-shot) and Step 12 (task variations) should test harder ones: very short messages ("ok"), emojis, questions with a complaint, messages in other languages, or clear praise + a minor complaint.
- **"Mixed → neutral" is a business decision, not a fact.** It makes the output consistent, but for customer service a message like "great sound, but uncomfortable" may need to be flagged as a complaint. Another option is a 4th label `mixed`. This should be agreed with TechFlow.
- **Temperature is still 0.7.** The prompt alone was enough to get 100%. For a production classifier, temperature 0 would add a safety margin.

**Conclusion:** clear instructions, a fixed label set with definitions, a single-word output format, and rules for the edge cases moved sentiment from **60% → 100%** consistency and from **0% → 100%** valid answers. The v2 prompt already meets the lab target (> 80% over 15 runs) for sentiment.

## Step 7: Improve Product Description Prompt

**What to do:**
1. Rewrite the product description prompt with:
   - Clear structure requirements
   - Length constraints
   - Style guidelines

In [22]:
#STEP 7: improve the product prompt (v2) to fix the failures found in Step 5

product_template_v2 = """You are a copywriter for TechFlow Solutions, an online store for consumer tech and accessories.

Write a product description for the product below.

Rules:
- Use ONLY the facts in the product information. Do not add specs, numbers, ports, battery types,
  range, materials or compatibility that are not given.
- If few details are given, describe the benefits of this type of product in general terms, without numbers.
- Use the product name exactly as given. Do not invent a brand or model name.

Structure (plain text, no markdown, no bold, no headings with #):
Line 1: <product name> – <price>
Then: one short paragraph (2 sentences) on who it is for and the main benefit.
Then the line "Key features:" followed by exactly 3 bullets starting with "- ".
Last line: one short call-to-action sentence.

Style: friendly and professional, address the reader as "you", no exclamation marks,
no superlatives (ultimate, best, revolutionary).

Length: 80–120 words in total.

Product information: {input}
"""

# Same setup as v1: 15 runs per test case, temperature 0.7 (only the prompt changes)
path = "results/step7_product_v2_15runs.csv"
if os.path.exists(path):   # re-use saved runs; delete the file to run again
    product_v2_15 = pd.read_csv(path)
else:
    product_v2_15 = run_test_cases(product_template_v2, product_cases, n=15, temperature=0.7)
    product_v2_15.to_csv(path, index=False)

product_v1_15 = pd.read_csv("results/step5_product_v1_15runs.csv")

def product_structure(text):
    """Signature of the v2 layout: title with price, 'Key features:' line, number of bullets, markdown used."""
    lines = [l.strip() for l in text.strip().split("\n") if l.strip()]
    bullets = sum(l.startswith("- ") for l in lines)
    has_md = "**" in text or "#" in text
    return f"price_in_title={'$' in lines[0]} | key_features={'Key features:' in lines} | bullets={bullets} | markdown={has_md}"

def product_v1_vs_v2(v1, v2):
    """Per product: structure consistency, length, invented specs and name kept, for v1 and v2."""
    rows = []
    for case, g2 in v2.groupby("input", sort=False):
        g1 = v1[v1["input"] == case]
        name = case.split(",")[0].strip().lower()
        w1, w2 = g1["output"].str.split().str.len(), g2["output"].str.split().str.len()
        s2 = g2["output"].map(product_structure)
        rows.append({
            "input": case[:50],
            "v1 consistency_% (header)": round(100 * g1["output"].map(header_style).value_counts().iloc[0] / len(g1)),
            "v2 consistency_% (structure)": round(100 * s2.value_counts().iloc[0] / len(s2)),
            "v2 most common structure": s2.value_counts().index[0],
            "v1 words": f"{w1.min()}-{w1.max()}",
            "v2 words": f"{w2.min()}-{w2.max()}",
            "v2 in 80-120 words_%": round(100 * w2.between(80, 120).mean()),
            "v1 valid_% (no invented specs)": round(100 * g1["output"].map(lambda o: is_valid("product", case, o)).mean()),
            "v2 valid_% (no invented specs)": round(100 * g2["output"].map(lambda o: is_valid("product", case, o)).mean()),
            "v2 name kept_%": round(100 * g2["output"].str.strip().str.lower().str.startswith(name).mean()),
            "v2 unique texts": g2["output"].nunique(),
        })
    return pd.DataFrame(rows)

product_comparison = product_v1_vs_v2(product_v1_15, product_v2_15)
product_comparison.to_csv("results/step7_product_v1_vs_v2.csv", index=False)
display(product_comparison)
print(f"Average consistency: v1 {product_comparison['v1 consistency_% (header)'].mean():.0f}% -> "
      f"v2 {product_comparison['v2 consistency_% (structure)'].mean():.0f}%")
print(f"Average valid (no invented specs): v1 {product_comparison['v1 valid_% (no invented specs)'].mean():.0f}% -> "
      f"v2 {product_comparison['v2 valid_% (no invented specs)'].mean():.0f}%")



,input,v1 consistency_% (header),v2 consistency_% (structure),v2 most common structure,v1 words,v2 words,v2 in 80-120 words_%,v1 valid_% (no invented specs),v2 valid_% (no invented specs),v2 name kept_%,v2 unique texts
0,"Wireless mouse, $29.99",93,100,price_in_title=True | key_features=True | bullets=3 | markdown=False,228-286,59-83,7,47,100,100,15
1,"USB-C hub, $49.99",47,100,price_in_title=True | key_features=True | bullets=3 | markdown=False,254-347,58-74,0,0,100,100,15
2,"Noise-cancelling headphones, $199, 30h battery, Bl",87,100,price_in_title=True | key_features=True | bullets=3 | markdown=False,205-286,60-81,13,93,100,100,15
3,"LED desk lamp with wireless charging pad, $59.99",60,100,price_in_title=True | key_features=True | bullets=3 | markdown=False,221-304,78-88,80,0,93,100,15


Average consistency: v1 72% -> v2 100%
Average valid (no invented specs): v1 35% -> v2 98%


### Step 7 Observations: product description v1 vs v2 (15 runs each)
Based on `results/step7_product_v2_15runs.csv`; the comparison table is saved as `results/step7_product_v1_vs_v2.csv`. Same 4 products, same model (`gpt-4o-mini`) and same temperature (0.7) as v1, so **only the prompt changed**.

**What changed in the prompt** (each change targets a failure from Step 5):

| Step 5 failure | v2 fix |
|---|---|
| P1 Hallucinated specs (hub and lamp 0% valid) | "Use ONLY the facts in the product information. Do not add specs, numbers, ports, battery types, range, materials or compatibility" |
| P2 Invented product names ("UltraConnect USB-C Hub") | "Use the product name exactly as given. Do not invent a brand or model name." |
| P3 Header style changes | Fixed structure: `<name> – <price>` title, 2-sentence intro, "Key features:" + exactly 3 bullets, call-to-action line; plain text, no markdown |
| P4 Length not controlled (205–347 words) | "Length: 80–120 words in total." |
| (style) No tone defined | Friendly and professional, address the reader as "you", no exclamation marks, no superlatives |
| P5 Every text is unique | Not targeted: creative text should vary; the **structure** is what must be consistent |

**How it is measured**
- **Consistency % (structure)**: share of runs with the most common layout signature (price in title, "Key features:" line, number of bullets, markdown used). For v1 the header style is used, as in Step 5.
- **Valid %**: no invented specs (same regex as Step 5).
- **Name kept %**: the first line starts with the product name as given.

**Results**

| Metric | v1 | v2 |
|---|---|---|
| Average consistency | 72% (header) | **100%** (structure) |
| Inputs above 80% | 2/4 | **4/4** |
| Valid (no invented specs) | 35% | **98%** |
| Product name kept as given | invented names | **100%** |
| Price in title | varied | **100%** |
| Words per description | 205–347 | **58–88** |
| Within 80–120 words | n/a | **25%** |
| Unique texts (of 15) | 15 | 15 |

- **All 60 answers have the same layout**: title with price, 2-sentence intro, "Key features:" with exactly 3 bullets, closing line, no markdown.
- **Hallucinations were mostly removed**: USB-C hub 0% → 100% valid, lamp 0% → 93% (1 run invented "adjustable brightness levels"). No invented brand names.
- **Style rules worked**: no exclamation marks and no banned superlatives in any run.

**What still fails**

| # | Problem | Evidence (15 runs) |
|---|---|---|
| P4' | **Too short: length rule not met** | 58–88 words; only 25% of runs are within 80–120 (hub 0%, mouse 7%, headphones 13%, lamp 80%). The fixed structure (2 sentences + 3 short bullets + 1 line) does not fit 80 words, so the model follows the structure and ignores the length. |
| P6 | **Given specs dropped (new, caused by v2)** | Headphones: "30h battery" and "Bluetooth 5.3" appear in only **3/15** runs and "3 colors" in **1/15**; "long-lasting battery life" replaces "30h" in 12/15. The rule "general terms, without numbers" was meant for inputs with few details, but the model applies it to every product. |
| P7 | **Soft hallucinations not caught by the regex** | "Plug-and-play" (hub, 10/15), "energy-efficient" (lamp, 9/15), "adjustable" lighting (lamp, 6/15), "compatible with various operating systems" (mouse). Claims that were not in the input. |
| P8 | **Generic wording** | "Ergonomic" in 14/15 mouse runs; clichés like "Take your … to the next level". |

**Limitations**
- The **valid % depends on the regex**: it only catches the specs listed in `INVENTED_SPECS`, so 98% is optimistic (see P7). Manual reading of the outputs is still needed.
- The 80–120 word range and the 3-bullet layout are **business choices** to agree with TechFlow.

**Next step → Step 11 (few-shot + structure):**
- Add a rule: "Include every fact given in the product information, with its exact number"; use general benefits only for facts that are *not* given.
- Make length and structure match (e.g. 60–90 words, or a 3–4 sentence intro).
- Add few-shot examples, including one product **with full specs** that shows every number kept.
- Extend `INVENTED_SPECS` with the soft claims and add a "given facts kept %" metric.

**Conclusion:** a fixed structure, a "facts only" rule and style guidelines moved product descriptions from **72% → 100%** layout consistency and from **35% → 98%** valid (no invented specs). The structure target is met, but the content now **leaves out facts that were given** and misses the length target, which few-shot examples should fix in Step 11.


## Step 8: Improve Data Extraction Prompt

**What to do:**
1. Rewrite the data extraction prompt with:
   - Structured output format
   - Specific fields to extract
   - Clear format requirements


In [23]:
#STEP 8: improve the extraction prompt (v2) to fix the failures found in Step 5
# Fixes: E1 no JSON -> JSON only | E2 field names drift -> fixed schema, all keys always present
#        E3 summary -> no text outside JSON | E4 interpretation -> only what is written
#        E5 value format -> digits only, dates as written, null if missing | E6 multiple items -> products list
extraction_template_v2 = """You extract structured data from TechFlow Solutions customer feedback.

Return ONLY a JSON object with exactly these keys (no other keys, no text before or after, no code fences):
{
  "order_number": string or null,
  "order_date": string or null,
  "delivery_date": string or null,
  "products": [{"name": string, "feedback": string or null}],
  "delivery_feedback": string or null,
  "packaging_feedback": string or null,
  "issues": [string],
  "customer_request": string or null
}

Rules:
- Extract only information written in the feedback. Do not add opinions, sentiment, advice or guesses.
- order_number: the order or item number, digits only, without "#".
- Dates: copy them exactly as written (e.g. "March 15th", "last Tuesday"). Do not convert them.
- order_date is when the customer ordered; delivery_date is when it arrived. Do not mix them.
- products: one object per product named in the text, name in lowercase. Use [] if no product is named.
- feedback, delivery_feedback, packaging_feedback: a few words taken from the text (e.g. "damaged", "fast").
- issues: short phrases for each problem the customer reports. Use [] if there are none.
- customer_request: what the customer wants or is waiting for (e.g. "refund"), otherwise null.
- If information is missing, use null (or [] for lists). Never leave a key out.

Customer feedback: "{input}"
"""

# Expected values for the fields that have one correct answer (used to check accuracy)
extraction_expected = {
    "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged.":
        {"order_number": "12345", "order_date": "march 15th", "delivery_date": None, "products": []},
    "My laptop stand arrived last Tuesday and one of the screws was missing.":
        {"order_number": None, "order_date": None, "delivery_date": "last tuesday", "products": ["laptop stand"]},
    "Order #88231: I bought a keyboard and a mouse. The keyboard is great, the mouse doesn't connect.":
        {"order_number": "88231", "order_date": None, "delivery_date": None, "products": ["keyboard", "mouse"]},
    "Still waiting for my refund. Nobody answers my emails!!":
        {"order_number": None, "order_date": None, "delivery_date": None, "products": []},
}
SCHEMA_KEYS = {"order_number", "order_date", "delivery_date", "products",
               "delivery_feedback", "packaging_feedback", "issues", "customer_request"}

# Same setup as v1: 15 runs per test case, temperature 0.7 (only the prompt changes)
path = "results/step8_extraction_v2_15runs.csv"
if os.path.exists(path):   # re-use saved runs; delete the file to run again
    extraction_v2_15 = pd.read_csv(path)
else:
    extraction_v2_15 = run_test_cases(extraction_template_v2, extraction_cases, n=15, temperature=0.7)
    extraction_v2_15.to_csv(path, index=False)

extraction_v1_15 = pd.read_csv("results/step5_extraction_v1_15runs.csv")

def parse_json(text):
    """Parsed JSON object, or None if the answer is not valid JSON."""
    try:
        obj = json.loads(re.sub(r"^```(?:json)?|```$", "", str(text).strip()).strip())
        return obj if isinstance(obj, dict) else None
    except ValueError:
        return None

def key_facts(obj):
    """The fields with one correct answer, normalised (lowercase, products sorted)."""
    low = lambda v: v.strip().lower() if isinstance(v, str) else v
    names = sorted(low(p.get("name")) for p in obj.get("products", []) if isinstance(p, dict))
    return {"order_number": low(obj.get("order_number")), "order_date": low(obj.get("order_date")),
            "delivery_date": low(obj.get("delivery_date")), "products": names}

def extraction_v1_vs_v2(v1, v2):
    """Per test case: valid JSON, exact schema, consistency of key facts / full answer, accuracy."""
    rows = []
    for case, g2 in v2.groupby("input", sort=False):
        g1 = v1[v1["input"] == case]
        parsed = g2["output"].map(parse_json)
        ok = parsed.dropna()
        facts = ok.map(lambda o: json.dumps(key_facts(o), sort_keys=True))
        full = ok.map(lambda o: json.dumps(o, sort_keys=True).lower())
        rows.append({
            "input": case[:50],
            "v1 consistency_% (field names)": round(100 * g1["output"].map(field_labels).value_counts().iloc[0] / len(g1)),
            "v1 valid JSON_%": round(100 * g1["output"].map(lambda o: is_valid("extraction", case, o)).mean()),
            "v2 valid JSON_%": round(100 * parsed.notna().mean()),
            "v2 exact schema_%": round(100 * ok.map(lambda o: set(o) == SCHEMA_KEYS).sum() / len(g2)),
            "v2 consistency_% (key facts)": round(100 * facts.value_counts().iloc[0] / len(g2)) if len(ok) else 0,
            "v2 consistency_% (full answer)": round(100 * full.value_counts().iloc[0] / len(g2)) if len(ok) else 0,
            "v2 accuracy_% (key facts)": round(100 * ok.map(lambda o: key_facts(o) == extraction_expected[case]).sum() / len(g2)),
            "v2 unique answers": g2["output"].nunique(),
        })
    return pd.DataFrame(rows)

extraction_comparison = extraction_v1_vs_v2(extraction_v1_15, extraction_v2_15)
extraction_comparison.to_csv("results/step8_extraction_v1_vs_v2.csv", index=False)
display(extraction_comparison)
for col in ["v1 consistency_% (field names)", "v2 consistency_% (key facts)", "v2 consistency_% (full answer)", "v2 valid JSON_%"]:
    print(f"{col}: {extraction_comparison[col].mean():.0f}%")


,input,v1 consistency_% (field names),v1 valid JSON_%,v2 valid JSON_%,v2 exact schema_%,v2 consistency_% (key facts),v2 consistency_% (full answer),v2 accuracy_% (key facts),v2 unique answers
0,I ordered item #12345 on March 15th. The delivery,60,0,100,100,100,67,100,3
1,My laptop stand arrived last Tuesday and one of th,40,0,100,100,100,93,100,2
2,Order #88231: I bought a keyboard and a mouse. The,53,0,100,100,100,100,100,3
3,Still waiting for my refund. Nobody answers my ema,33,0,100,100,100,53,100,4


v1 consistency_% (field names): 46%
v2 consistency_% (key facts): 100%
v2 consistency_% (full answer): 78%
v2 valid JSON_%: 100%


In [ ]:
#STEP 8 CHECKPOINT: compare v1 and v2 for all three prompts
s = pd.read_csv("results/step6_sentiment_v1_vs_v2.csv")
p = pd.read_csv("results/step7_product_v1_vs_v2.csv")
e = pd.read_csv("results/step8_extraction_v1_vs_v2.csv")

def row(task, v1, v2, valid1, valid2):
    return {"task": task,
            "v1 avg consistency_%": round(v1.mean()), "v2 avg consistency_%": round(v2.mean()),
            "v1 inputs > 80%": f"{(v1 > 80).sum()}/{len(v1)}", "v2 inputs > 80%": f"{(v2 > 80).sum()}/{len(v2)}",
            "v1 valid_%": round(valid1.mean()), "v2 valid_%": round(valid2.mean())}

checkpoint = pd.DataFrame([
    row("sentiment", s["v1 consistency_%"], s["v2 consistency_%"], s["v1 valid_%"], s["v2 valid_%"]),
    row("product", p["v1 consistency_% (header)"], p["v2 consistency_% (structure)"],
        p["v1 valid_% (no invented specs)"], p["v2 valid_% (no invented specs)"]),
    row("extraction", e["v1 consistency_% (field names)"], e["v2 consistency_% (key facts)"],
        e["v1 valid JSON_%"], e["v2 valid JSON_%"]),
])
checkpoint.to_csv("results/step8_v1_vs_v2_all.csv", index=False)
display(checkpoint)
print(f"Extraction full-JSON consistency (incl. free-text 'issues'): {e['v2 consistency_% (full answer)'].mean():.0f}%")


### Step 8 Observations: extraction v1 vs v2 (15 runs each)
Based on `results/step8_extraction_v2_15runs.csv`; the comparison table is saved as `results/step8_extraction_v1_vs_v2.csv`. Same 4 test cases, same model (`gpt-4o-mini`) and same temperature (0.7) as v1, so **only the prompt changed**.

**What changed in the prompt** (each change targets a failure from Step 5):

| Step 5 failure | v2 fix |
|---|---|
| E1 No machine-readable format (0/60 JSON) | "Return ONLY a JSON object… no text before or after, no code fences" |
| E2 Field names drift (up to 9 field sets per input) | Fixed schema with 8 exact keys, all always present |
| E3 Summary instead of fields | No text allowed outside the JSON |
| E4 Interpretation added as data ("Frustration", "Sentiment") | "Extract only information written in the feedback. Do not add opinions, sentiment, advice or guesses." |
| E5 Value format varies (`#12345` vs `12345`, "last Tuesday") | Digits only for numbers, dates copied exactly as written, `null` when missing, order date vs delivery date defined |
| E6 Structure changes with two items | `products` is always a list with one object per product |

**How it is measured**
- **Valid JSON % / exact schema %**: answer parses as JSON and has exactly the 8 keys.
- **Consistency % (key facts)**: share of runs that give the most common value for the fields with one correct answer (`order_number`, `order_date`, `delivery_date`, product names).
- **Consistency % (full answer)**: share of runs where the *whole* JSON is identical, including free-text fields (`issues`, `feedback`).
- **Accuracy % (key facts)**: key facts match the expected values.

**Results**

| Metric | v1 | v2 |
|---|---|---|
| Valid JSON | 0% | **100%** |
| Exact schema | n/a | **100%** |
| Consistency (key facts / field names for v1) | 46% | **100%** |
| Consistency (full JSON) | n/a | **78%** |
| Inputs above 80% (key facts) | 0/4 | **4/4** |
| Accuracy (key facts) | n/a | **100%** |
| Unique answers per input (of 15) | 11–15 | **2–4** |

- All 60 answers are valid JSON with the exact schema, and every key fact is correct.
- The order number is always `"12345"` (no `#`), and "last Tuesday" is always the `delivery_date` (in v1 it was once put under "Order Date").
- The two-item order (#88231) gave the **same JSON in 15/15 runs**: one object per product, each with its own feedback.

**What still varies: only the `issues` field**

| Input | Full-JSON consistency | What varies |
|---|---|---|
| Baseline (#12345) | 67% | "packaging damaged" (10) vs "damaged packaging" (3) vs **empty list** (2) |
| Laptop stand | 93% | "one of the screws was missing" (14) vs "screw missing" (1) |
| Two items (#88231) | 100% | nothing |
| Refund complaint | 53% | "Nobody answers my emails" **missing from `issues` in 8/15 runs**; "waiting for refund" only sometimes listed |

- **Missed problems (recall)** are the main remaining failure. The model sees the packaging damage (`packaging_feedback: "damaged"`) but sometimes does not list it as an issue, and it often skips the unanswered emails.
- **Wording varies** in free text ("packaging damaged" vs "damaged packaging"), which is expected.
- **Unclear rule:** the prompt does not say whether something the customer is waiting for (refund) is also an issue, or only the `customer_request`.

**Next step → Step 10 (Chain-of-Thought):** this is a reasoning problem, not a format problem. The model should first go through the feedback sentence by sentence and list every problem, then fill in the JSON. Add a definition: "an issue is any problem the customer reports, including delivery, packaging, product or customer support."

---

### Checkpoint: all three prompts, v1 vs v2 (15 runs, temperature 0.7)
Summary table saved as `results/step8_v1_vs_v2_all.csv`.

| Prompt | Consistency v1 → v2 | Inputs > 80% v1 → v2 | Valid v1 → v2 | Main remaining issue |
|---|---|---|---|---|
| Sentiment | 60% → **100%** (exact label) | 1/5 → **5/5** | 0% → **100%** (single allowed label) | Only 5 test messages; "mixed → neutral" is a business choice |
| Product description | 72% → **100%** (layout) | 2/4 → **4/4** | 35% → **98%** (no invented specs) | Too short (only 25% in 80–120 words); given specs dropped (headphones "30h"/"Bluetooth 5.3" in 3/15) |
| Data extraction | 46% → **100%** (key facts) / **78%** (full JSON) | 0/4 → **4/4** | 0% → **100%** (valid JSON) | `issues` incomplete (refund case 53%) |


## Step 9: Add Few-Shot Examples to Sentiment Analysis

**What to do:**
1. Add 2-3 examples showing the exact format you want
2. Test the few-shot prompt 15 times

**Why extra test messages are needed:** v2 already gave 100% on the 5 original messages, so few-shot cannot show any gain there. The v3 prompt is therefore tested on the 5 original messages **and** on 7 harder messages taken from the Step 6 limitations (very short text, emoji, question with a problem, praise + small complaint, sarcasm without obvious words, another language, a neutral question). v2 is run on the same hard messages so the comparison is fair.

**Rule for the examples:** none of the few-shot examples is one of the test messages (otherwise the model could simply copy the answer). They cover each label and each edge case once.


In [24]:
#STEP 9: sentiment v3 = v2 instructions + few-shot examples
import os
import pandas as pd

_needed = ["sentiment_cases", "sentiment_expected", "run_test_cases", "sentiment_template_v2"]
_missing = [n for n in _needed if n not in globals()]
if _missing:
    raise RuntimeError(f"Run the cells above first (Steps 1-6). Missing: {', '.join(_missing)}")

sentiment_template_v3 = """You are a sentiment classifier for TechFlow Solutions customer messages.

Classify the sentiment of the customer message below.

Allowed labels (use exactly one):
- positive: the customer is satisfied or happy with the product or service
- negative: the customer is unhappy, complains, or reports a problem
- neutral: the customer states facts or asks something without expressing an opinion

Rules:
- If the message is sarcastic, classify what the customer really means, not the literal words.
- If the message contains both praise and a complaint, answer neutral.
- A question that reports a problem is negative; a question without a problem is neutral.
- Messages can be in any language or contain emojis; classify the meaning.

Output format: reply with only the label in lowercase, as a single word. No punctuation, no explanation.

Examples:
Message: "The battery lasts all day, really happy with it."
Label: positive

Message: "Third time contacting support and the screen still flickers."
Label: negative

Message: "The case looks nice, but it scratches easily."
Label: neutral

Message: "Yeah, great job sending me the wrong color."
Label: negative

Message: "Does the charger come with a cable?"
Label: neutral

Message: "{input}"
Label:"""

# Harder messages (not in the examples, not in the original test set) with the expected label
sentiment_hard_expected = {
    "ok": "neutral",                                                            # very short
    "Works perfectly 👍": "positive",                                           # emoji
    "How do I return this? It doesn't fit my laptop.": "negative",              # question + problem
    "Love it, although the cable is a bit short.": "neutral",                   # praise + small complaint (rule)
    "Thanks for nothing.": "negative",                                          # sarcasm without obvious words
    "El teclado no funciona desde que llegó.": "negative",                      # other language (Spanish)
    "Is this mouse compatible with a MacBook Air?": "neutral",                  # question, no problem
}
sentiment_hard_cases = list(sentiment_hard_expected)
all_expected = {**sentiment_expected, **sentiment_hard_expected}

def run_or_load_csv(path, template, cases):
    """15 runs per case at temperature 0.7 (same as v1/v2); re-use saved runs, delete the file to run again."""
    if os.path.exists(path):
        return pd.read_csv(path)
    df = run_test_cases(template, cases, n=15, temperature=0.7)
    df.to_csv(path, index=False)
    return df

sentiment_v3_15 = run_or_load_csv("results/step9_sentiment_v3_15runs.csv", sentiment_template_v3, sentiment_cases)
sentiment_v2_hard = run_or_load_csv("results/step9_sentiment_v2_hard_15runs.csv", sentiment_template_v2, sentiment_hard_cases)
sentiment_v3_hard = run_or_load_csv("results/step9_sentiment_v3_hard_15runs.csv", sentiment_template_v3, sentiment_hard_cases)
sentiment_v2_15 = pd.read_csv("results/step6_sentiment_v2_15runs.csv")

def label_stats(df, case):
    """Consistency, validity and accuracy of the exact (lowercased) answers for one message."""
    a = df.loc[df["input"] == case, "output"].str.strip().str.lower()
    return {"consistency_%": round(100 * a.value_counts().iloc[0] / len(a)),
            "valid_%": round(100 * a.isin(["positive", "negative", "neutral"]).mean()),
            "accuracy_%": round(100 * (a == all_expected[case]).mean()),
            "answers": {k: int(v) for k, v in a.value_counts().items()}}

def sentiment_v2_vs_v3(v2, v3, cases, test_set):
    rows = []
    for case in cases:
        s2, s3 = label_stats(v2, case), label_stats(v3, case)
        rows.append({"test set": test_set, "input": case[:55], "expected": all_expected[case],
                     "v2 consistency_%": s2["consistency_%"], "v3 consistency_%": s3["consistency_%"],
                     "v2 accuracy_%": s2["accuracy_%"], "v3 accuracy_%": s3["accuracy_%"],
                     "v2 valid_%": s2["valid_%"], "v3 valid_%": s3["valid_%"],
                     "v2 answers": s2["answers"], "v3 answers": s3["answers"]})
    return rows

sentiment_v2_v3 = pd.DataFrame(
    sentiment_v2_vs_v3(sentiment_v2_15, sentiment_v3_15, sentiment_cases, "original")
    + sentiment_v2_vs_v3(sentiment_v2_hard, sentiment_v3_hard, sentiment_hard_cases, "hard"))
sentiment_v2_v3.to_csv("results/step9_sentiment_v2_vs_v3.csv", index=False)
display(sentiment_v2_v3)

summary = sentiment_v2_v3.groupby("test set", sort=False).agg(
    v2_consistency=("v2 consistency_%", "mean"), v3_consistency=("v3 consistency_%", "mean"),
    v2_accuracy=("v2 accuracy_%", "mean"), v3_accuracy=("v3 accuracy_%", "mean"),
    v2_above_80=("v2 consistency_%", lambda s: f"{(s > 80).sum()}/{len(s)}"),
    v3_above_80=("v3 consistency_%", lambda s: f"{(s > 80).sum()}/{len(s)}"),
).round()
display(summary)


,test set,input,expected,v2 consistency_%,v3 consistency_%,v2 accuracy_%,v3 accuracy_%,v2 valid_%,v3 valid_%,v2 answers,v3 answers
0,original,I love this product! It's exactly what I needed.,positive,100,100,100,100,100,100,{'positive': 15},{'positive': 15}
1,original,The keyboard stopped working after a week. Very disappo,negative,100,100,100,100,100,100,{'negative': 15},{'negative': 15}
2,original,"Great sound quality, but the headphones are uncomfortab",neutral,100,100,100,100,100,100,{'neutral': 15},{'neutral': 15}
3,original,"Oh wonderful, my new charger already broke. Just what I",negative,100,100,100,100,100,100,{'negative': 15},{'negative': 15}
4,original,I received the webcam today. Haven't tried it yet.,neutral,100,100,100,100,100,100,{'neutral': 15},{'neutral': 15}
5,hard,ok,neutral,100,100,100,100,100,100,{'neutral': 15},{'neutral': 15}
6,hard,Works perfectly 👍,positive,100,100,100,100,100,100,{'positive': 15},{'positive': 15}
7,hard,How do I return this? It doesn't fit my laptop.,negative,100,100,0,100,100,100,{'neutral': 15},{'negative': 15}
8,hard,"Love it, although the cable is a bit short.",neutral,100,100,100,100,100,100,{'neutral': 15},{'neutral': 15}
9,hard,Thanks for nothing.,negative,100,100,100,100,100,100,{'negative': 15},{'negative': 15}


,v2_consistency,v3_consistency,v2_accuracy,v3_accuracy,v2_above_80,v3_above_80
test set,,,,,,
original,100.0,100.0,100.0,100.0,5/5,5/5
hard,100.0,100.0,86.0,100.0,7/7,7/7


### Step 9 Observations: sentiment v2 vs v3 few-shot (15 runs each)
Based on `results/step9_sentiment_v3_15runs.csv`, `results/step9_sentiment_v2_hard_15runs.csv` and `results/step9_sentiment_v3_hard_15runs.csv`; the comparison table is saved as `results/step9_sentiment_v2_vs_v3.csv`. Same model (`gpt-4o-mini`) and temperature (0.7) as v1 and v2.

**What changed in the prompt (v2 → v3)**

| Change | Why |
|---|---|
| 5 few-shot examples in a `Message:` / `Label:` format, ending with `Label:` | Shows the exact output format; one example per label plus one for mixed and one for sarcasm |
| New rule: "A question that reports a problem is negative; a question without a problem is neutral." | Questions were not covered by any v2 rule |
| New rule: "Messages can be in any language or contain emojis; classify the meaning." | Edge cases listed in the Step 6 limitations |

The instructions ask for 2-3 examples; 5 were used (the troubleshooting section recommends 3-5) so that each label and each edge case appears once. **None of the examples is a test message**, so the model cannot score by copying an answer.

**Results**

| Test set | v2 consistency | v3 consistency | v2 accuracy | v3 accuracy | Inputs > 80% (v3) |
|---|---|---|---|---|---|
| Original (5 messages) | 100% | **100%** | 100% | **100%** | 5/5 |
| Hard (7 messages) | 100% | **100%** | 86% | **100%** | 7/7 |

- **The only difference** is "How do I return this? It doesn't fit my laptop.": v2 answered `neutral` in **15/15** runs, v3 answers `negative` in **15/15**.
- Every other hard message (very short "ok", emoji, Spanish, sarcasm "Thanks for nothing.", praise + small complaint, neutral question) already got the same correct answer with v2.

**Key finding: consistency is not correctness.** On the return question v2 was **100% consistent and 100% wrong**. A consistency metric alone would have reported it as perfect; only checking against an expected label showed the error. This is why every step from Step 6 measures accuracy as well as consistency.

**Limitations**
- **The gain cannot be credited to few-shot alone.** v3 added the examples *and* the question rule at the same time, and either could have fixed the return question. A clean test would run v2 + the new rule without examples on that message (an ablation). Here few-shot did not hurt anything and kept the format at 100%.
- The hard set has only 7 messages, and the expected labels are my own judgement; "praise + small complaint → neutral" is still the business rule from Step 6.

**Conclusion:** v3 keeps **100% consistency** on all 12 messages and raises accuracy on the hard set from **86% → 100%**. For a simple 3-label task, clear rules and label definitions (v2) did most of the work; few-shot examples and an extra rule fixed the remaining edge case.


## Step 10: Add Chain-of-Thought to Data Extraction

**What to do:**
1. Add Chain-of-Thought reasoning for complex extraction tasks
2. Test the CoT prompt 15 times

**What v3 targets** (the failures left after Step 8): the model **missed issues** (refund case: "Nobody answers my emails" missing in 8/15 runs; baseline: damaged packaging missing in 2/15), the **wording of issues varied** ("packaging damaged" vs "damaged packaging"), and it was **unclear** whether something the customer is still waiting for (a refund) is an issue.

**How:** the model first reasons step by step (sentences → facts → issues → request) under `Reasoning:`, then gives the JSON under `JSON:`. Only the JSON part is parsed and compared. A new metric, **issues correct %**, checks that every expected problem is listed and that nothing positive is listed as a problem.


In [ ]:
#STEP 10: extraction v3 = v2 schema + Chain-of-Thought (reason step by step, then the JSON)
import os, re, json
import pandas as pd

_needed = ["extraction_cases", "run_test_cases", "extraction_expected", "SCHEMA_KEYS", "parse_json", "key_facts"]
_missing = [n for n in _needed if n not in globals()]
if _missing:
    raise RuntimeError(f"Run the cells above first (Steps 1-8). Missing: {', '.join(_missing)}")

extraction_template_v3 = """You extract structured data from TechFlow Solutions customer feedback.

First think step by step, then give the JSON. Use exactly this format:

Reasoning:
1. Sentences: list each sentence of the feedback.
2. Facts: for each sentence, write the facts it gives (order number, dates, products, delivery, packaging).
3. Issues: for each sentence, write any problem the customer reports. An issue is any problem with a product,
   the delivery, the packaging, a refund or customer support, including something the customer is still waiting for.
4. Request: write what the customer wants, if anything.

JSON:
<the JSON object>

The JSON object must have exactly these keys (no other keys, no code fences, nothing after the JSON):
{
  "order_number": string or null,
  "order_date": string or null,
  "delivery_date": string or null,
  "products": [{"name": string, "feedback": string or null}],
  "delivery_feedback": string or null,
  "packaging_feedback": string or null,
  "issues": [string],
  "customer_request": string or null
}

JSON rules:
- Extract only information written in the feedback. Do not add opinions, sentiment, advice or guesses.
- order_number: the order or item number, digits only, without "#".
- Dates: copy them exactly as written (e.g. "March 15th", "last Tuesday"). Do not convert them.
- order_date is when the customer ordered; delivery_date is when it arrived. Do not mix them.
- products: one object per product named in the text, name in lowercase. Use [] if no product is named.
- feedback, delivery_feedback, packaging_feedback: a few words taken from the text (e.g. "damaged", "fast").
- issues: one entry for every problem found in step 3, in lowercase, written as "<thing> <problem>"
  (e.g. "packaging damaged", "mouse not connecting", "refund not received"). Use [] only if step 3 found no problem.
- customer_request: what the customer wants or is waiting for (e.g. "refund"), otherwise null.
- If information is missing, use null (or [] for lists). Never leave a key out.

Customer feedback: "{input}"
"""

# Problems that must be in "issues" (all patterns must match) and things that must NOT be listed as a problem
ISSUES_EXPECTED = {
    "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged.": [r"packag"],
    "My laptop stand arrived last Tuesday and one of the screws was missing.": [r"screw"],
    "Order #88231: I bought a keyboard and a mouse. The keyboard is great, the mouse doesn't connect.": [r"mouse|connect"],
    "Still waiting for my refund. Nobody answers my emails!!": [r"refund", r"email|answer|respon|support|contact"],
}
NOT_ISSUES = {
    "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged.": [r"deliver"],
    "Order #88231: I bought a keyboard and a mouse. The keyboard is great, the mouse doesn't connect.": [r"keyboard"],
}

def parse_cot_json(text):
    """The JSON object after the last 'JSON:' marker (the reasoning before it is ignored), or None."""
    part = str(text).rsplit("JSON:", 1)[-1]
    start, end = part.find("{"), part.rfind("}")
    if start == -1 or end == -1:
        return None
    try:
        obj = json.loads(part[start:end + 1])
        return obj if isinstance(obj, dict) else None
    except ValueError:
        return None

def issues_correct(obj, case, expected=ISSUES_EXPECTED, not_issues=NOT_ISSUES):
    """True if every expected problem is in 'issues' and nothing positive is listed as a problem."""
    issues = obj.get("issues") or []
    text = " | ".join(str(i) for i in issues).lower()
    found_all = all(re.search(p, text) for p in expected.get(case, []))
    no_false = not any(re.search(p, text) for p in not_issues.get(case, []))
    return found_all and no_false and (bool(issues) or not expected.get(case))

path = "results/step10_extraction_v3_15runs.csv"
if os.path.exists(path):   # re-use saved runs; delete the file to run again
    extraction_v3_15 = pd.read_csv(path)
else:
    extraction_v3_15 = run_test_cases(extraction_template_v3, extraction_cases, n=15, temperature=0.7)
    extraction_v3_15.to_csv(path, index=False)
extraction_v2_15 = pd.read_csv("results/step8_extraction_v2_15runs.csv")

def extraction_scores(df, case, parser, expected_facts):
    """Metrics for one input; only the parsed JSON is compared (not the reasoning)."""
    g = df[df["input"] == case]
    ok = g["output"].map(parser).dropna()
    n = len(g)
    facts = ok.map(lambda o: json.dumps(key_facts(o), sort_keys=True))
    full = ok.map(lambda o: json.dumps(o, sort_keys=True).lower())
    return {
        "valid JSON_%": round(100 * len(ok) / n),
        "exact schema_%": round(100 * ok.map(lambda o: set(o) == SCHEMA_KEYS).sum() / n),
        "consistency_% (key facts)": round(100 * facts.value_counts().iloc[0] / n) if len(ok) else 0,
        "consistency_% (full JSON)": round(100 * full.value_counts().iloc[0] / n) if len(ok) else 0,
        "accuracy_% (key facts)": round(100 * ok.map(lambda o: key_facts(o) == expected_facts).sum() / n),
        "issues correct_%": round(100 * ok.map(lambda o: issues_correct(o, case)).sum() / n),
        "avg output words": round(g["output"].str.split().str.len().mean()),
    }

rows = []
for case in extraction_cases:
    s2 = extraction_scores(extraction_v2_15, case, parse_json, extraction_expected[case])
    s3 = extraction_scores(extraction_v3_15, case, parse_cot_json, extraction_expected[case])
    row = {"input": case[:50]}
    for k in s2:
        row[f"v2 {k}"], row[f"v3 {k}"] = s2[k], s3[k]
    row["v3 reasoning shown_%"] = round(100 * extraction_v3_15.loc[extraction_v3_15["input"] == case, "output"]
                                        .str.contains("Reasoning:").mean())
    rows.append(row)

extraction_v2_v3 = pd.DataFrame(rows)
extraction_v2_v3.to_csv("results/step10_extraction_v2_vs_v3.csv", index=False)
display(extraction_v2_v3.T)
for k in ["consistency_% (key facts)", "consistency_% (full JSON)", "issues correct_%", "valid JSON_%", "avg output words"]:
    print(f"{k}: v2 {extraction_v2_v3[f'v2 {k}'].mean():.0f} -> v3 {extraction_v2_v3[f'v3 {k}'].mean():.0f}")

# The issues found by v3, per input (to read the free-text field by hand)
for case in extraction_cases:
    issues = extraction_v3_15.loc[extraction_v3_15["input"] == case, "output"].map(parse_cot_json).dropna() \
             .map(lambda o: json.dumps(o.get("issues"))).value_counts()
    print(f"\n{case[:60]}\n{issues.to_string()}")


## Step 11: Add Few-Shot Examples and Structure to Product Description

**What to do:**
1. Combine few-shot examples with structured output format
2. Test the combined approach 15 times

**What v3 targets** (the failures left after Step 7): **given specs dropped** (headphones "30h battery" / "Bluetooth 5.3" in only 3/15 runs), **too short** (only 25% within 80–120 words, because the structure was too small for the length), **soft hallucinations** the regex did not catch ("plug-and-play", "energy-efficient", "adjustable"), and **generic wording** ("take your … to the next level").

**How:** two few-shot examples for products that are **not** in the test set: one with full specs (shows every number kept) and one with almost no details (shows general benefits without inventing features). The structure is made larger (3-sentence intro, full-sentence bullets) so it matches an 80–110 word target. Two new metrics: **given facts kept %** and **strict valid %** (Step 5 regex + the soft claims found in Step 7).


In [ ]:
#STEP 11: product v3 = v2 structure + "keep every given fact" + few-shot examples
import os, re
import pandas as pd

_needed = ["product_cases", "run_test_cases", "INVENTED_SPECS", "product_structure", "header_style"]
_missing = [n for n in _needed if n not in globals()]
if _missing:
    raise RuntimeError(f"Run the cells above first (Steps 1-7). Missing: {', '.join(_missing)}")

product_template_v3 = """You are a copywriter for TechFlow Solutions, an online store for consumer tech and accessories.

Write a product description for the product below.

Rules:
- Include EVERY fact given in the product information, with its exact number or value
  (for example "30h battery", "Bluetooth 5.3", "3 colors").
- Do not add any other fact: no specs, numbers, ports, battery types, range, materials, compatibility,
  "plug-and-play", energy efficiency or adjustable settings unless they are given.
- If few facts are given, describe the everyday benefits of this type of product in general terms, without adding features.
- Use the product name exactly as given. Do not invent a brand or model name.

Structure (plain text, no markdown, no bold, no headings with #):
Line 1: <product name> – <price>
Then: an intro of 3 sentences: who it is for, the main benefit, and how it fits into their day.
Then the line "Key features:" followed by exactly 3 bullets starting with "- ", each one a full sentence.
If more than 3 facts are given, combine facts in a bullet so that every fact appears.
Last line: one short call-to-action sentence.

Style: friendly and professional, address the reader as "you", no exclamation marks,
no superlatives (ultimate, best, revolutionary), no clichés (such as "take it to the next level").

Length: 80–110 words in total.

Example 1 (many facts given: every fact is kept)
Product information: Portable SSD, $89.99, 1TB, USB-C, 1050MB/s
Description:
Portable SSD – $89.99
This portable SSD is made for students, creators and professionals who move large files every day. With 1TB of storage and speeds of up to 1050MB/s, you spend less time waiting and more time working. It slips easily into a bag, so your work comes with you.
Key features:
- 1TB of storage gives you room for projects, photos and backups in one place.
- Transfer speeds of up to 1050MB/s help you move large files quickly.
- A USB-C connection keeps your setup simple with a single cable.
Keep your files fast and close at hand with this portable SSD.

Example 2 (few facts given: general benefits, no invented features)
Product information: Phone stand, $14.99
Description:
Phone stand – $14.99
This phone stand is for anyone who wants to keep their phone in view while working, cooking or watching videos. It holds your phone upright, so you can read messages and follow calls without picking it up. It is a small addition that makes your desk tidier and your day easier.
Key features:
- Keeps your phone upright so the screen is always easy to see.
- Frees your hands for typing, cooking or taking notes.
- A compact size that fits on a desk, a shelf or a kitchen counter.
Give your phone a steady place on your desk with this phone stand.

Now write the description. Reply with the description only, starting with line 1.
Product information: {input}
Description:
"""

# Facts that were given and must appear (regex, lowercase text)
GIVEN_FACTS = {
    "Wireless mouse, $29.99": [r"wireless", r"\$29\.99"],
    "USB-C hub, $49.99": [r"usb-c", r"\$49\.99"],
    "Noise-cancelling headphones, $199, 30h battery, Bluetooth 5.3, foldable, 3 colors":
        [r"noise.cancel", r"\$199", r"30.?h|30 hours|30-hour", r"bluetooth 5\.3", r"fold", r"(?:3|three) colou?rs"],
    "LED desk lamp with wireless charging pad, $59.99": [r"\bled\b", r"wireless charging", r"\$59\.99"],
}
# Soft claims found in Step 7 that the Step 5 regex did not catch
SOFT_CLAIMS = {
    "Wireless mouse, $29.99": r"compatib|operating system|plug.and.play",
    "USB-C hub, $49.99": r"plug.and.play|compatib|durable",
    "Noise-cancelling headphones, $199, 30h battery, Bluetooth 5.3, foldable, 3 colors": r"compatib|quick charg|fast charg",
    "LED desk lamp with wireless charging pad, $59.99": r"energy.efficien|adjustable",
}
CLICHES = r"next level|game.changer|elevate your"

def facts_kept(text, patterns):
    t = str(text).lower()
    return sum(bool(re.search(p, t)) for p in patterns) / len(patterns)

def strict_valid(text, case):
    t = str(text).lower()
    return not re.search(INVENTED_SPECS[case], t) and not re.search(SOFT_CLAIMS[case], t)

path = "results/step11_product_v3_15runs.csv"
if os.path.exists(path):   # re-use saved runs; delete the file to run again
    product_v3_15 = pd.read_csv(path)
else:
    product_v3_15 = run_test_cases(product_template_v3, product_cases, n=15, temperature=0.7)
    product_v3_15.to_csv(path, index=False)
product_v1_15 = pd.read_csv("results/step5_product_v1_15runs.csv")
product_v2_15 = pd.read_csv("results/step7_product_v2_15runs.csv")

def product_scores(g, case, structure_fn, lo, hi):
    out, words = g["output"], g["output"].str.split().str.len()
    s = out.map(structure_fn)
    name = case.split(",")[0].strip().lower()
    return {
        "consistency_%": round(100 * s.value_counts().iloc[0] / len(s)),
        "words": f"{words.min()}-{words.max()}",
        "in length range_%": round(100 * words.between(lo, hi).mean()) if lo else None,
        "valid_% (Step 5 regex)": round(100 * out.map(lambda o: not re.search(INVENTED_SPECS[case], o.lower())).mean()),
        "strict valid_% (+ soft claims)": round(100 * out.map(lambda o: strict_valid(o, case)).mean()),
        "given facts kept_%": round(100 * out.map(lambda o: facts_kept(o, GIVEN_FACTS[case])).mean()),
        "name kept_%": round(100 * out.str.strip().str.lower().str.startswith(name).mean()),
        "cliches_%": round(100 * out.str.lower().str.contains(CLICHES).mean()),
    }

rows = []
for case in product_cases:
    sel = lambda df: df[df["input"] == case]
    scores = {"v1": product_scores(sel(product_v1_15), case, header_style, None, None),
              "v2": product_scores(sel(product_v2_15), case, product_structure, 80, 120),
              "v3": product_scores(sel(product_v3_15), case, product_structure, 80, 110)}
    row = {"input": case[:50]}
    for metric in scores["v3"]:
        for v in ["v1", "v2", "v3"]:
            row[f"{v} {metric}"] = scores[v][metric]
    row["v3 most common structure"] = sel(product_v3_15)["output"].map(product_structure).value_counts().index[0]
    rows.append(row)

product_v1_v2_v3 = pd.DataFrame(rows)
product_v1_v2_v3.to_csv("results/step11_product_v1_v2_v3.csv", index=False)
display(product_v1_v2_v3.T)
print("(v1 consistency = header style; v2/v3 = layout signature. Length target: v2 80-120, v3 80-110 words.)")
for metric in ["consistency_%", "in length range_%", "strict valid_% (+ soft claims)", "given facts kept_%", "cliches_%"]:
    vals = {v: product_v1_v2_v3[f"{v} {metric}"].dropna() for v in ["v1", "v2", "v3"]}
    print(f"{metric}: " + " -> ".join(f"{v} {s.mean():.0f}" if len(s) else f"{v} n/a" for v, s in vals.items()))


## Step 12: Create Task Variations

**What to do:**
1. Create variations of each task
2. Test your best prompts on these variations
3. Document what works and what needs adjustment

The best prompt for each task (v3) is run **unchanged** on new inputs it has never seen, 15 runs each at temperature 0.7. Each variation targets something the original test cases did not cover:

| Task | Variation | What it tests |
|---|---|---|
| Sentiment | "Not bad at all, I'd buy it again." | negation |
| Sentiment | "WORST PURCHASE EVER. DO NOT BUY." | all caps / shouting |
| Sentiment | "😡😡😡" | emoji only, no words |
| Sentiment | long review: praise + a solved problem + "very pleased" | long text where the overall opinion is clear but the mixed rule may apply |
| Product | Smartwatch with 5 specs | more than 3 facts for 3 bullets |
| Product | "Gaming chair" (no price) | missing price: does it invent one? |
| Product | Laptop sleeve with sizes and colors | non-technical facts |
| Product | Webcam, 1080p, microphone | short input with one key spec |
| Extraction | order never arrived, numeric date "02/11/2026" | non-arrival + refund request + date format |
| Extraction | two products, two dates, one missing | partial delivery |
| Extraction | Spanish feedback | other language |
| Extraction | only praise | no issues at all: issues must be [] |


In [ ]:
#STEP 12: test the v3 prompts on task variations (prompts unchanged)
import os, re, json
import pandas as pd

_needed = ["sentiment_template_v3", "product_template_v3", "extraction_template_v3", "run_test_cases",
           "product_structure", "parse_cot_json", "key_facts", "issues_correct", "facts_kept"]
_missing = [n for n in _needed if n not in globals()]
if _missing:
    raise RuntimeError(f"Run the cells above first (Steps 1-11). Missing: {', '.join(_missing)}")

sentiment_variations = {
    "Not bad at all, I'd buy it again.": "positive",                                     # negation
    "WORST PURCHASE EVER. DO NOT BUY.": "negative",                                     # all caps
    "😡😡😡": "negative",                                                                 # emoji only
    ("I've had the speaker for two months. Setup was easy and the sound is clear. "
     "The app crashed once, but support fixed it within a day. Overall I'm very pleased."): "positive",  # long, overall clear
}
product_variations = {   # given facts that must appear
    "Smartwatch, $149, heart-rate monitor, GPS, 7-day battery, water-resistant to 50m":
        [r"\$149", r"heart.rate", r"\bgps\b", r"7.day|7 days|seven.day|seven days", r"water.resistant", r"50 ?m"],
    "Gaming chair": [r"gaming chair"],
    "Laptop sleeve, $24.99, fits 13-14 inch laptops, available in black and grey":
        [r"\$24\.99", r"13", r"14", r"black", r"gr[ae]y"],
    "Webcam, $69.99, 1080p, built-in microphone": [r"\$69\.99", r"1080p", r"microphone"],
}
extraction_variations = {   # expected key facts, issue patterns that must be found, patterns that must not
    "Hi, order 55120 placed on 02/11/2026 never arrived. I want my money back.": (
        {"order_number": "55120", "order_date": "02/11/2026", "delivery_date": None, "products": []},
        [r"arriv|deliver|receiv"], []),
    "Bought headphones and a charger (order #7781) on Monday. The headphones arrived Wednesday, the charger is still missing.": (
        {"order_number": "7781", "order_date": "monday", "delivery_date": "wednesday", "products": ["charger", "headphones"]},
        [r"charger"], [r"headphones"]),
    "Pedí el pedido #3321 el 5 de mayo y llegó roto.": (
        {"order_number": "3321", "order_date": "5 de mayo", "delivery_date": None, "products": []},
        [r"rot|broke|damag"], []),
    "Great service, thanks! The monitor works perfectly.": (
        {"order_number": None, "order_date": None, "delivery_date": None, "products": ["monitor"]},
        [], [r"."]),   # any issue is wrong here
}

def run_or_load(path, template, cases):
    if os.path.exists(path):
        return pd.read_csv(path)
    df = run_test_cases(template, cases, n=15, temperature=0.7)
    df.to_csv(path, index=False)
    return df

sent_var = run_or_load("results/step12_sentiment_variations_15runs.csv", sentiment_template_v3, list(sentiment_variations))
prod_var = run_or_load("results/step12_product_variations_15runs.csv", product_template_v3, list(product_variations))
extr_var = run_or_load("results/step12_extraction_variations_15runs.csv", extraction_template_v3, list(extraction_variations))

rows = []
for case, expected in sentiment_variations.items():
    a = sent_var.loc[sent_var["input"] == case, "output"].str.strip().str.lower()
    rows.append({"task": "sentiment", "input": case[:60],
                 "consistency_%": round(100 * a.value_counts().iloc[0] / len(a)),
                 "accuracy / quality_%": round(100 * (a == expected).mean()),
                 "details": f"expected {expected}; answers {({k: int(v) for k, v in a.value_counts().items()})}"})

for case, facts in product_variations.items():
    out = prod_var.loc[prod_var["input"] == case, "output"]
    s = out.map(product_structure)
    words = out.str.split().str.len()
    invented_price = out.str.contains(r"\$").mean() if "$" not in case else 0
    rows.append({"task": "product", "input": case[:60],
                 "consistency_%": round(100 * s.value_counts().iloc[0] / len(s)),
                 "accuracy / quality_%": round(100 * out.map(lambda o: facts_kept(o, facts)).mean()),
                 "details": f"given facts kept; words {words.min()}-{words.max()} "
                            f"({round(100 * words.between(80, 110).mean())}% in 80-110); "
                            f"structure: {s.value_counts().index[0]}"
                            + (f"; invented a price in {round(100 * invented_price)}% of runs" if "$" not in case else "")})

for case, (facts, must, must_not) in extraction_variations.items():
    ok = extr_var.loc[extr_var["input"] == case, "output"].map(parse_cot_json).dropna()
    n = (extr_var["input"] == case).sum()
    kf = ok.map(lambda o: json.dumps(key_facts(o), sort_keys=True))
    issues_ok = ok.map(lambda o: issues_correct(o, case, {case: must}, {case: must_not}))
    rows.append({"task": "extraction", "input": case[:60],
                 "consistency_%": round(100 * kf.value_counts().iloc[0] / n) if len(ok) else 0,
                 "accuracy / quality_%": round(100 * ok.map(lambda o: key_facts(o) == facts).sum() / n),
                 "details": f"key facts accuracy; valid JSON {round(100 * len(ok) / n)}%; "
                            f"issues correct {round(100 * issues_ok.sum() / n)}%; "
                            f"most common key facts {kf.value_counts().index[0] if len(ok) else '-'}"})

variations = pd.DataFrame(rows)
variations.to_csv("results/step12_variations.csv", index=False)
pd.set_option("display.max_colwidth", None)
display(variations)
print("accuracy / quality = exact label (sentiment), given facts kept (product), key facts correct (extraction)")


## Step 13: Final Evaluation and Comparison

**What to do:**
1. Run your final prompts (version 3) 15 times each
2. Create a comprehensive comparison report
3. Document improvement metrics

The v3 prompts were already run 15 times on every original test case in Steps 9-11 (`results/step9_*`, `step10_*`, `step11_*`), with the same model and temperature as v1 and v2. This cell re-uses those runs and puts v1, v2 and v3 side by side for all three tasks.


In [ ]:
#STEP 13: final comparison v1 -> v2 -> v3 (15 runs each, temperature 0.7, same original test cases)
import re, json
import pandas as pd

_needed = ["classified_as", "header_style", "field_labels", "is_valid", "product_structure", "parse_json",
           "parse_cot_json", "key_facts", "issues_correct", "strict_valid", "facts_kept", "GIVEN_FACTS",
           "sentiment_expected", "extraction_expected", "SCHEMA_KEYS"]
_missing = [n for n in _needed if n not in globals()]
if _missing:
    raise RuntimeError(f"Run the cells above first (Steps 1-12). Missing: {', '.join(_missing)}")

runs = {
    "sentiment": {"v1": "results/step5_sentiment_v1_15runs.csv", "v2": "results/step6_sentiment_v2_15runs.csv",
                  "v3": "results/step9_sentiment_v3_15runs.csv"},
    "product": {"v1": "results/step5_product_v1_15runs.csv", "v2": "results/step7_product_v2_15runs.csv",
                "v3": "results/step11_product_v3_15runs.csv"},
    "extraction": {"v1": "results/step5_extraction_v1_15runs.csv", "v2": "results/step8_extraction_v2_15runs.csv",
                   "v3": "results/step10_extraction_v3_15runs.csv"},
}

def pct(series):
    return round(100 * series.mean())

def top_share(series):
    return 100 * series.value_counts().iloc[0] / len(series)

def evaluate(task, version, df):
    """Per input: consistency (main metric) and quality metrics; returned as averages over the inputs."""
    rows = []
    for case, g in df.groupby("input", sort=False):
        out = g["output"]
        if task == "sentiment":
            label = out.map(classified_as) if version == "v1" else out.str.strip().str.lower()
            rows.append({"consistency": top_share(label),
                         "valid": pct(out.map(lambda o: is_valid("sentiment", case, o))),
                         "accuracy": pct(label == sentiment_expected[case]) if version != "v1" else 0})
        elif task == "product":
            shape = out.map(header_style if version == "v1" else product_structure)
            rows.append({"consistency": top_share(shape),
                         "valid": pct(out.map(lambda o: strict_valid(o, case))),
                         "accuracy": pct(out.map(lambda o: facts_kept(o, GIVEN_FACTS[case])))})
        else:
            if version == "v1":
                rows.append({"consistency": top_share(out.map(field_labels)), "valid": 0, "accuracy": 0, "issues": 0})
                continue
            parsed = out.map(parse_json if version == "v2" else parse_cot_json)
            ok = parsed.dropna()
            facts = ok.map(lambda o: json.dumps(key_facts(o), sort_keys=True))
            rows.append({"consistency": 100 * facts.value_counts().iloc[0] / len(out) if len(ok) else 0,
                         "valid": 100 * len(ok) / len(out),
                         "accuracy": 100 * ok.map(lambda o: key_facts(o) == extraction_expected[case]).sum() / len(out),
                         "issues": 100 * ok.map(lambda o: issues_correct(o, case)).sum() / len(out)})
    r = pd.DataFrame(rows)
    return {"avg consistency_%": round(r["consistency"].mean()),
            "inputs > 80%": f"{(r['consistency'] > 80).sum()}/{len(r)}",
            "valid_%": round(r["valid"].mean()),
            "accuracy_%": round(r["accuracy"].mean()),
            **({"issues correct_%": round(r["issues"].mean())} if task == "extraction" else {})}

final = []
for task, versions in runs.items():
    for version, path in versions.items():
        final.append({"task": task, "version": version, **evaluate(task, version, pd.read_csv(path))})
final = pd.DataFrame(final)
final.to_csv("results/step13_final_comparison.csv", index=False)
display(final)

print("""How to read it:
- consistency: sentiment = same label; product = same layout (v1: header style); extraction = same key facts (v1: field names)
- valid: sentiment = single allowed label; product = no invented specs incl. soft claims; extraction = parseable JSON
- accuracy: sentiment = correct label; product = given facts kept; extraction = key facts correct (v1 never valid -> 0)""")
